# 17 · Capstone Projects

**Part VII — Engineering, trust and practice** · Estimated time: ~90 min (plus project time)

Sixteen notebooks have taught you the pieces: tensors and autograd, optimizers, convolutions,
recurrent networks, attention, pretrained Transformers, generative models, reinforcement learning,
efficient training, deployment and trust. What they could not teach, because it only happens when you
own a problem end to end, is *judgement*: which metric to optimize, how to know whether a result is
real, when a model is good enough, and how to tell others what you found. This closing notebook is
about that. It has three layers. The first is a short course on running a deep learning project, with a
checklist you can reuse. The second is code: a compact but complete `Trainer` class with configuration,
logging, checkpointing, evaluation and early stopping, demonstrated on a small task, plus helpers for
result tables and plots. The third is a set of six fully specified project briefs (and an optional
seventh, a paper reproduction), each with data suggestions, milestones, stretch goals, a grading rubric
and a runnable starter cell. The notebook closes with guidance on reproducibility and reporting, a
reading list, and a glossary of the course's vocabulary.

## Learning objectives

- Frame a deep learning problem in terms of a decision, a metric, a baseline and constraints, and audit a dataset before modelling.
- Build and use a reusable `Trainer` with a configuration dataclass, a training history, JSONL logging, checkpoint/resume and early stopping.
- Choose one of six capstone projects, plan it with milestones, and start it from the provided starter code.
- Design experiments whose conclusions survive scrutiny: baselines, ablations, multiple seeds, error analysis.
- Report results reproducibly, including negative results, with plots that communicate and a report structure that readers expect.
- Write a model card and a one-page project plan.
- Know where to continue: books, courses, key papers, communities, and how to read a paper.

## Prerequisites

- Notebook 05 · The PyTorch Workflow and Best Practices (the `fit`/`evaluate` pattern, checkpoints, reproducibility).
- The notebooks behind the project you choose: Notebook 07 · Transfer Learning and Modern Vision, Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs, Notebook 09 · Attention and Transformers, Notebook 10 · NLP with Pretrained Transformers (Hugging Face), Notebook 12 · GANs and Diffusion Models, Notebook 13 · Reinforcement Learning.
- Notebook 15 · Deployment and Model Optimization (model cards) and Notebook 16 · Interpretability, Robustness and Uncertainty (calibration, error analysis).

## Contents

1. How to run a deep learning project
2. Project template code: a reusable `Trainer`
3. Project briefs (a)–(g) with starter code
4. Reproducibility and reporting
5. What next: reading list, communities, glossary
6. Exercises · Summary · Further reading

In [ ]:
import math, os, random, time, json, sys, platform, hashlib, copy, re
from dataclasses import dataclass, asdict, field, replace
from pathlib import Path
from collections import Counter
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = DATA_DIR / "runs"                     # every experiment writes its artefacts here
RUNS_DIR.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(DATA_DIR / "torch_hub"))   # torchvision weights download into DATA_DIR too

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# optional libraries used by individual project starters (imported inside try/except)
try:
    import gymnasium as gym
    GYM_AVAILABLE = True
except ImportError:
    gym = None
    GYM_AVAILABLE = False
    print("gymnasium is not installed -> project (f) uses a built-in gridworld (see environment.yml)")
try:
    import transformers
    from transformers import AutoModelForSequenceClassification
    transformers.logging.set_verbosity_error()
    HF_AVAILABLE = True
except ImportError:
    transformers = AutoModelForSequenceClassification = None
    HF_AVAILABLE = False
    print("transformers is not installed -> project (e) uses an in-notebook model (see environment.yml)")


def seed_everything(seed=SEED):
    """Seed Python, NumPy and PyTorch (CPU and CUDA)."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))

## 1. How to run a deep learning project

### 1.1 Problem framing: decision, metric, baseline, constraints

Most failed projects fail before the first line of model code, at the framing stage. Four questions
settle it. **What decision will the model support?** A model that predicts "customer will churn" is
useless unless someone can act on the prediction in time; a segmentation model for a microscope is
only valuable if it is faster or more consistent than the technician it assists. Write the decision
down in one sentence. **Which metric captures the decision?** Accuracy is the default and it is
usually wrong: for imbalanced classes you want precision/recall, F1 or the area under the
precision–recall curve; for forecasting, an error in the units the user cares about (MAE in degrees,
not MSE in squared degrees); for generation, a human-judged quality measure supplemented by an
automatic proxy. Choose *one* primary metric to optimize and a few secondary metrics to monitor, and
choose them before you see any results. **What is the baseline?** The majority class, a linear model,
last week's value, the heuristic the domain expert uses today. The baseline is what your model must
beat *convincingly*, and it is astonishing how often a well-tuned linear model is hard to beat.
**What are the constraints?** Latency, memory, the hardware you can afford, the amount of labelled
data, privacy rules, and the deadline. A constraint discovered in week six wastes weeks one to five.

> **Key idea.** A deep learning project is an experiment with a hypothesis ("model X will beat
> baseline Y on metric Z by a margin that matters for decision W"); write the hypothesis first.

### 1.2 The data audit

Before modelling, spend an hour looking at the data as if you distrusted it, because you should.
Count the examples per class or per value range. Look at random examples *and* at the extremes. Check
for exact and near duplicates, especially across the train/validation split, because leakage between
the two is the most common cause of results that are too good to be true. Check label quality on a
random sample of a hundred items by hand; a 5% label-noise rate caps the accuracy you can measure.
Look at the distribution of inputs (image sizes, sequence lengths, value ranges) to choose
preprocessing, and ask where the data came from and whether the deployment data will look the same.
We make this concrete with the small dataset that serves as the running example of the whole
notebook: a subset of FashionMNIST (downloaded with torchvision into `DATA_DIR`, with a synthetic
fallback of geometric shapes if the download is impossible).

In [ ]:
def make_synthetic_shapes(n, seed=SEED):
    """Fallback dataset: 28x28 images of four shapes (filled square, hollow square, disc, plus sign)."""
    rng = np.random.default_rng(seed)
    X = np.zeros((n, 1, 28, 28), dtype=np.float32)
    y = rng.integers(0, 4, size=n)
    yy, xx = np.mgrid[0:28, 0:28]
    for i in range(n):
        cx, cy, r = rng.integers(9, 19), rng.integers(9, 19), rng.integers(4, 8)
        if y[i] == 0:
            mask = (abs(xx - cx) <= r) & (abs(yy - cy) <= r)
        elif y[i] == 1:
            mask = (abs(xx - cx) <= r) & (abs(yy - cy) <= r) & ~((abs(xx - cx) <= r - 2) & (abs(yy - cy) <= r - 2))
        elif y[i] == 2:
            mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= r * r
        else:
            mask = ((abs(xx - cx) <= 1) & (abs(yy - cy) <= r)) | ((abs(yy - cy) <= 1) & (abs(xx - cx) <= r))
        X[i, 0] = mask * rng.uniform(0.6, 1.0) + rng.normal(0, 0.08, size=(28, 28))
    X = np.clip(X, 0, 1)
    return torch.tensor(X), torch.tensor(y, dtype=torch.long)


def load_demo_data(n_train, n_val):
    """FashionMNIST subsets as float tensors in [0, 1], or synthetic shapes if the download fails."""
    try:
        from torchvision import datasets
        tr = datasets.FashionMNIST(str(DATA_DIR), train=True, download=True)
        te = datasets.FashionMNIST(str(DATA_DIR), train=False, download=True)
        g = torch.Generator().manual_seed(SEED)
        itr = torch.randperm(len(tr), generator=g)[:n_train]
        ite = torch.randperm(len(te), generator=g)[:n_val]
        Xtr = tr.data[itr].float().div(255).unsqueeze(1); ytr = tr.targets[itr]
        Xva = te.data[ite].float().div(255).unsqueeze(1); yva = te.targets[ite]
        return Xtr, ytr, Xva, yva, list(tr.classes), "FashionMNIST"
    except Exception as e:
        print(f"FashionMNIST unavailable ({type(e).__name__}: {str(e)[:80]}) -> synthetic shapes dataset")
        Xtr, ytr = make_synthetic_shapes(n_train, SEED)
        Xva, yva = make_synthetic_shapes(n_val, SEED + 1)
        return Xtr, ytr, Xva, yva, ["filled square", "hollow square", "disc", "plus"], "synthetic shapes"


N_TRAIN = 3000 if FAST_RUN else 20000
N_VAL = 1000 if FAST_RUN else 5000
X_train, y_train, X_val, y_val, class_names, data_source = load_demo_data(N_TRAIN, N_VAL)
n_classes = len(class_names)
print(f"{data_source}: train {tuple(X_train.shape)}, val {tuple(X_val.shape)}, {n_classes} classes")

The audit function below does the checks listed above that can be automated: shapes, value ranges,
class balance, exact duplicates within the training set, and exact duplicates *between* train and
validation (leakage). It hashes the raw bytes of each example, which catches exact copies only; for
near-duplicates you would compare embeddings or perceptual hashes. It also shows a random sample, since
no summary statistic replaces looking.

In [ ]:
def audit_dataset(X_train, y_train, X_val, y_val, class_names, show=10):
    def hashes(X):
        return [hashlib.md5(x.cpu().numpy().tobytes()).hexdigest() for x in X]
    print(f"train: {tuple(X_train.shape)} {X_train.dtype}, range [{X_train.min():.2f}, {X_train.max():.2f}], "
          f"mean {X_train.mean():.3f}, std {X_train.std():.3f}")
    print(f"val  : {tuple(X_val.shape)}")
    counts = Counter(y_train.tolist())
    print("class balance (train):", {class_names[k][:12]: v for k, v in sorted(counts.items())})
    print(f"majority-class share: {max(counts.values()) / len(y_train):.3f}")
    h_tr, h_va = hashes(X_train), hashes(X_val)
    print(f"exact duplicates within train: {len(h_tr) - len(set(h_tr))}")
    print(f"train/val leakage (identical examples): {len(set(h_tr) & set(h_va))}")
    fig, axes = plt.subplots(1, show, figsize=(1.2 * show, 1.6))
    idx = torch.randperm(len(X_train), generator=torch.Generator().manual_seed(0))[:show]
    for ax, i in zip(axes, idx):
        ax.imshow(X_train[i, 0], cmap="gray"); ax.set_title(class_names[y_train[i]][:8], fontsize=7); ax.axis("off")
    plt.suptitle("random training examples", fontsize=9); plt.tight_layout(); plt.show()


audit_dataset(X_train, y_train, X_val, y_val, class_names)

FashionMNIST is balanced (10% per class); in our subsets the audit finds at most a handful of exact
duplicates and no identical train/validation images, and the pixels are already normalized to
$[0, 1]$. Real datasets rarely pass an audit this cleanly.

### 1.3 Baselines first

Two baselines take a minute to compute and calibrate your expectations for everything that follows.
The **majority-class** baseline is the accuracy of always predicting the most frequent class; it is the
floor, and for imbalanced data it is often embarrassingly high. The **nearest-centroid** classifier
(predict the class whose mean image is closest in Euclidean distance) is the simplest model that uses
the inputs at all. Section 2 adds a linear model and a CNN with the `Trainer`; the point of recording
these numbers now is that every later result is judged against them.

In [ ]:
majority = Counter(y_train.tolist()).most_common(1)[0][0]
majority_acc = (y_val == majority).float().mean().item()

centroids = torch.stack([X_train[y_train == c].mean(0) for c in range(n_classes)])       # (C, 1, 28, 28)
dists = torch.cdist(X_val.flatten(1), centroids.flatten(1))                               # (N_val, C)
centroid_acc = (dists.argmin(1) == y_val).float().mean().item()
baseline_results = {"majority class": majority_acc, "nearest centroid": centroid_acc}
print(f"majority class ('{class_names[majority]}') accuracy: {majority_acc:.3f}")
print(f"nearest-centroid accuracy: {centroid_acc:.3f}")

### 1.4 Iterate with experiments, then analyse errors, then stop

With the framing, the audit and the baselines in place, the modelling loop is: make *one* change,
run it, log it, compare against the previous best on the primary metric, and keep or discard. Changes
that are worth making early are the ones with the largest expected effect: more or cleaner data, a
pretrained backbone, the right loss and metric, a sane learning-rate schedule. Architecture tweaks and
exotic optimizers come last. Two disciplines separate productive iteration from thrashing. First, an
**experiment log** (Section 4) in which every run has a name, a configuration, a seed, a result and a
one-line conclusion, so that you never re-run an experiment because you forgot its outcome. Second,
**error analysis**: look at the examples the model gets wrong, grouped by class, by input property
(length, brightness, source) and by confidence. The errors tell you what to fix next far more reliably
than the aggregate metric does; a model that fails only on one class needs data for that class, not a
bigger network. Section 2.5 demonstrates the two standard tools, the confusion matrix and the list of
most-confident mistakes.

**When to stop.** Stop when the primary metric has met the target that the decision requires, when the
last few changes moved it by less than the run-to-run noise (Section 4.2 shows how to measure that
noise), or when the remaining gap can only be closed with resources you do not have. "The model could
be a little better" is always true and is not a reason to continue.

### 1.5 The project checklist

The following checklist condenses this section; the exercises ask you to fill it in for your project.

1. **Framing**: decision supported, primary metric, secondary metrics, target value, constraints.
2. **Data**: source and license, sizes, splits (and *why* the split is valid: time-based, group-based, random), audit results, known issues.
3. **Baselines**: trivial baseline, simple model, prior work or existing system, with numbers.
4. **Plan**: milestones with dates, the first experiment, the ablations that matter, the stretch goals.
5. **Infrastructure**: the training harness, the experiment log, seeds, versions, checkpoints.
6. **Iteration**: one change at a time, error analysis after every significant result.
7. **Validation**: several seeds, a held-out test set touched once, sanity checks (train on a subset, overfit one batch, shuffle labels).
8. **Reporting**: results table with variance, plots, limitations, negative results, model card.

## 2. Project template code: a reusable `Trainer`

### 2.1 Design

Every notebook of this course has re-implemented a `train_one_epoch`/`evaluate` pair. For a project
that runs dozens of experiments you want the same logic once, tested, with the bookkeeping that
experiments need. The `Trainer` below is deliberately small (about a hundred lines) so that you can
read it in full and adapt it, but complete in the sense that it covers what real projects require:

- a **`TrainConfig` dataclass** holding every hyperparameter, saved as `config.json` in the run
  directory, so that a run can be reproduced from its folder alone;
- a **history** (a list of per-epoch dictionaries) that is also appended to `log.jsonl`, one JSON
  object per line, a format that survives crashes and is trivial to load with pandas;
- **checkpoints**: `last.pt` after every epoch (model, optimizer, scheduler, epoch, history, RNG
  states) and `best.pt` whenever the monitored metric improves, plus `fit(resume=True)` to continue an
  interrupted run;
- **evaluation** through a pluggable `loss_fn` and `metric_fn`, so that the same class handles
  classification (accuracy) and regression (MAE) in the project starters;
- **early stopping** with a patience counter on the monitored metric;
- **hooks** (`train_step`, `on_epoch_end`) that subclasses can override (Exercise 4).

It takes a model and two DataLoaders that yield `(inputs, targets)` pairs; anything more exotic
(dictionaries of inputs, several losses) is a small change to `train_step` and `predict`.

In [ ]:
@dataclass
class TrainConfig:
    """Everything that defines a run. Saved as JSON next to the checkpoints."""
    run_name: str = "run"
    epochs: int = 5
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-4
    optimizer: str = "adamw"        # "adamw" or "sgd"
    scheduler: str = "cosine"       # "cosine" or "none"
    grad_clip: float = 1.0          # 0 disables clipping
    patience: int = 3               # early-stopping patience in epochs; 0 disables
    monitor: str = "val_acc"        # key of the history row to monitor
    mode: str = "max"               # "max" or "min"
    seed: int = SEED
    notes: str = ""


def accuracy_metric(logits, targets):
    return {"acc": (logits.argmax(1) == targets).float().mean().item()}


class Trainer:
    """Minimal but complete training harness: config, history, JSONL log, checkpoints, early stopping."""

    def __init__(self, model, cfg, train_loader, val_loader, loss_fn=F.cross_entropy,
                 metric_fn=accuracy_metric, root=RUNS_DIR, verbose=True):
        self.model, self.cfg = model.to(device), cfg
        self.train_loader, self.val_loader = train_loader, val_loader
        self.loss_fn, self.metric_fn, self.verbose = loss_fn, metric_fn, verbose
        self.run_dir = Path(root) / cfg.run_name
        self.run_dir.mkdir(parents=True, exist_ok=True)
        (self.run_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")
        self.log_path = self.run_dir / "log.jsonl"
        self.optimizer = self._make_optimizer()
        self.scheduler = self._make_scheduler()
        self.history, self.step_losses = [], []
        self.epoch, self.best_score, self.bad_epochs = 0, None, 0

    # ----- construction helpers -------------------------------------------------------------
    def _make_optimizer(self):
        params = [p for p in self.model.parameters() if p.requires_grad]
        if self.cfg.optimizer == "sgd":
            return torch.optim.SGD(params, lr=self.cfg.lr, momentum=0.9, weight_decay=self.cfg.weight_decay)
        return torch.optim.AdamW(params, lr=self.cfg.lr, weight_decay=self.cfg.weight_decay)

    def _make_scheduler(self):
        if self.cfg.scheduler == "cosine":
            return torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=max(1, self.cfg.epochs))
        return None

    # ----- one step / one epoch / evaluation -------------------------------------------------
    def train_step(self, xb, yb):
        """One optimizer step; returns the loss value. Override to customize (e.g. mixed precision)."""
        logits = self.model(xb)
        loss = self.loss_fn(logits, yb)
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if self.cfg.grad_clip:
            nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
        self.optimizer.step()
        return loss.item()

    def train_one_epoch(self):
        self.model.train()
        total, n = 0.0, 0
        for xb, yb in self.train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = self.train_step(xb, yb)
            self.step_losses.append(loss)
            total += loss * xb.size(0); n += xb.size(0)
        return total / n

    @torch.no_grad()
    def predict(self, loader):
        """Return (outputs, targets) for a whole loader, on the CPU."""
        self.model.eval()
        outs, targets = [], []
        for xb, yb in loader:
            outs.append(self.model(xb.to(device)).cpu()); targets.append(yb)
        return torch.cat(outs), torch.cat(targets)

    def evaluate(self, loader=None):
        outs, targets = self.predict(loader or self.val_loader)
        result = {"loss": self.loss_fn(outs, targets).item()}
        result.update(self.metric_fn(outs, targets))
        return result

    # ----- the fit loop ----------------------------------------------------------------------
    def _is_better(self, score):
        if self.best_score is None:
            return True
        return score > self.best_score if self.cfg.mode == "max" else score < self.best_score

    def fit(self, resume=False):
        if resume and (self.run_dir / "last.pt").exists():
            self.load_checkpoint(self.run_dir / "last.pt")
            self._say(f"resumed from epoch {self.epoch}")
        else:
            seed_everything(self.cfg.seed)
            self.log_path.write_text("", encoding="utf-8")
        t_start = time.time()
        while self.epoch < self.cfg.epochs:
            t0 = time.time()
            lr = self.optimizer.param_groups[0]["lr"]
            train_loss = self.train_one_epoch()
            if self.scheduler is not None:
                self.scheduler.step()
            val = self.evaluate()
            self.epoch += 1
            row = {"epoch": self.epoch, "train_loss": train_loss,
                   **{f"val_{k}": v for k, v in val.items()}, "lr": lr, "time": time.time() - t0}
            self.log(row)
            if self._is_better(row[self.cfg.monitor]):
                self.best_score, self.bad_epochs = row[self.cfg.monitor], 0
                self.save_checkpoint(self.run_dir / "best.pt")
            else:
                self.bad_epochs += 1
            self.save_checkpoint(self.run_dir / "last.pt")
            self.on_epoch_end(row)
            if self.cfg.patience and self.bad_epochs >= self.cfg.patience:
                self._say(f"early stopping after epoch {self.epoch} (no improvement for {self.cfg.patience} epochs)")
                break
        self._say(f"done: best {self.cfg.monitor} = {self.best_score:.4f} | {time.time() - t_start:.1f}s | {self.run_dir}")
        return self.history

    def on_epoch_end(self, row):
        """Hook for subclasses; called after each epoch with the logged row."""

    # ----- logging and checkpoints -----------------------------------------------------------
    def _say(self, msg):
        if self.verbose:
            print(f"[{self.cfg.run_name}] {msg}")

    def log(self, row):
        self.history.append(row)
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row) + "\n")
        vals = " | ".join(f"{k[4:]} {v:.4f}" for k, v in row.items() if k.startswith("val_"))
        self._say(f"epoch {row['epoch']:3d} | train loss {row['train_loss']:.4f} | val {vals} | "
                  f"lr {row['lr']:.1e} | {row['time']:.1f}s")

    def save_checkpoint(self, path):
        rng = {"torch": torch.get_rng_state(), "numpy": np.random.get_state(), "python": random.getstate(),
               "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
        torch.save({"model": self.model.state_dict(), "optimizer": self.optimizer.state_dict(),
                    "scheduler": self.scheduler.state_dict() if self.scheduler else None,
                    "epoch": self.epoch, "history": self.history, "best_score": self.best_score,
                    "bad_epochs": self.bad_epochs, "config": asdict(self.cfg), "rng": rng}, path)

    def load_checkpoint(self, path, model_only=False):
        ck = torch.load(path, map_location="cpu", weights_only=False)
        self.model.load_state_dict(ck["model"])
        if model_only:
            return
        self.optimizer.load_state_dict(ck["optimizer"])
        if self.scheduler is not None and ck["scheduler"] is not None:
            self.scheduler.load_state_dict(ck["scheduler"])
            if self.cfg.epochs != ck["config"]["epochs"]:
                # The epoch budget changed on resume. A cosine schedule is defined by its total length
                # (after the old budget it has decayed the lr to zero), so rebuild it for the new budget
                # at the current epoch instead of continuing the finished one.
                self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    self.optimizer, T_max=max(1, self.cfg.epochs), last_epoch=ck["epoch"] - 1)
        self.epoch, self.history = ck["epoch"], ck["history"]
        self.best_score, self.bad_epochs = ck["best_score"], ck["bad_epochs"]
        torch.set_rng_state(ck["rng"]["torch"]); np.random.set_state(ck["rng"]["numpy"]); random.setstate(ck["rng"]["python"])
        if ck["rng"].get("cuda") is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(ck["rng"]["cuda"])


print("Trainer defined")

### 2.2 Result tables and plots

Two small helpers complete the template. `results_table` summarizes a dictionary of finished trainers
(best value of the monitored metric, the epoch at which it occurred, parameter count, total time) and
returns the rows so that you can also render them as markdown for a report. `plot_histories` overlays
the validation metric of several runs, which is the plot you will look at most often.

In [ ]:
def results_table(trainers, metric="val_acc", higher_is_better=True):
    """Print a comparison table for {name: Trainer} and return the rows as dicts."""
    rows = []
    for name, tr in trainers.items():
        pick = max if higher_is_better else min
        best = pick(tr.history, key=lambda r: r[metric])
        rows.append({"run": name, "params": count_params(tr.model), f"best {metric}": best[metric],
                     "epoch": best["epoch"], "final train loss": tr.history[-1]["train_loss"],
                     "time (s)": sum(r["time"] for r in tr.history)})
    print(f"{'run':22s} {'params':>10s} {'best ' + metric:>14s} {'epoch':>6s} {'train loss':>11s} {'time (s)':>9s}")
    for r in rows:
        print(f"{r['run']:22s} {r['params']:10,d} {r[f'best {metric}']:14.4f} {r['epoch']:6d} "
              f"{r['final train loss']:11.4f} {r['time (s)']:9.1f}")
    return rows


def plot_histories(trainers, metric="val_acc", title=None):
    for name, tr in trainers.items():
        plt.plot([r["epoch"] for r in tr.history], [r[metric] for r in tr.history], marker="o", label=name)
    plt.xlabel("epoch"); plt.ylabel(metric); plt.title(title or f"{metric} per epoch"); plt.legend()
    plt.tight_layout(); plt.show()


def make_loaders(X_tr, y_tr, X_va, y_va, batch_size, n_train=None, n_val=None):
    """DataLoaders over tensors, optionally on subsets (used by the fast demos)."""
    X_tr, y_tr = (X_tr[:n_train], y_tr[:n_train]) if n_train else (X_tr, y_tr)
    X_va, y_va = (X_va[:n_val], y_va[:n_val]) if n_val else (X_va, y_va)
    pin = (device.type == "cuda")
    tr = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin,
                    generator=torch.Generator().manual_seed(SEED))
    va = DataLoader(TensorDataset(X_va, y_va), batch_size=256, shuffle=False, num_workers=0, pin_memory=pin)
    return tr, va


train_loader, val_loader = make_loaders(X_train, y_train, X_val, y_val, batch_size=64)
print(f"{len(train_loader)} training batches, {len(val_loader)} validation batches")

### 2.3 Demonstration: linear baseline versus a small CNN

We now run the two models that Section 1.3 promised: a linear classifier on the flattened pixels and
a small convolutional network with BatchNorm and dropout. Both use the same `Trainer`, the same data
and the same configuration except for the learning rate, the schedule, the patience and the number of
epochs, which is how a comparison should be set up. Under `FAST_RUN` the budgets are tiny (a few epochs on 3,000 images), so
treat the absolute numbers as a smoke test; the full run trains on 20,000 images and reaches about 90%
validation accuracy with the CNN.

In [ ]:
class SmallCNN(nn.Module):
    """Two conv blocks + MLP head for 1x28x28 inputs (about 0.4M parameters at width 32)."""

    def __init__(self, num_classes, width=32, use_bn=True, dropout=0.3):
        super().__init__()
        norm = (lambda c: nn.BatchNorm2d(c)) if use_bn else (lambda c: nn.Identity())
        self.features = nn.Sequential(
            nn.Conv2d(1, width, 3, padding=1), norm(width), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(width, 2 * width, 3, padding=1), norm(2 * width), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(2 * width * 7 * 7, 128), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x):
        return self.head(self.features(x))


EPOCHS_DEMO = 3 if FAST_RUN else 12
seed_everything(SEED)
linear_model = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, n_classes))
cfg_linear = TrainConfig(run_name="baseline_linear", epochs=min(EPOCHS_DEMO, 5), lr=1e-3, scheduler="none",
                         patience=0, notes="linear baseline on raw pixels")
t_linear = Trainer(linear_model, cfg_linear, train_loader, val_loader)
t_linear.fit()

seed_everything(SEED)
cfg_cnn = TrainConfig(run_name="small_cnn", epochs=EPOCHS_DEMO, lr=2e-3, patience=3, notes="SmallCNN width 32, BN, dropout 0.3")
t_cnn = Trainer(SmallCNN(n_classes), cfg_cnn, train_loader, val_loader)
t_cnn.fit()

rows = results_table({"linear": t_linear, "small CNN": t_cnn})
print("baselines:", {k: round(v, 3) for k, v in baseline_results.items()})
plot_histories({"linear": t_linear, "small CNN": t_cnn}, title="Validation accuracy: linear vs CNN")

The ordering is the expected one, majority class < nearest centroid < linear < CNN, and the size of the
gaps is informative: the linear model already captures most of what pixels can tell us about clothing
categories, and the CNN adds a further few points by exploiting local structure. Every run has left a
folder under `data/runs/<run_name>` with `config.json`, `log.jsonl`, `best.pt` and `last.pt`; Section 4
looks inside.

### 2.4 Checkpointing, resuming and loading the best model

Training jobs get interrupted: by a pre-emptible cloud instance, by a laptop lid, by a bug in the
plotting code after the epochs finished. `fit(resume=True)` picks up from `last.pt`, restoring the
model, optimizer and scheduler states, the epoch counter, the history and the random-number generator
states, so that the continued run behaves like the run that would have happened without the
interruption (up to hardware non-determinism, and up to the DataLoader's own shuffling generator,
whose state the checkpoint does not include; save it too if you need bit-exact resumption). The
demonstration trains for two epochs, then constructs a *fresh* trainer with the same `run_name` and a
larger epoch budget and resumes. Extending the budget has one consequence worth knowing: a cosine
schedule is defined by its total length, and after two of two epochs it has already decayed the
learning rate to zero, so `load_checkpoint` rebuilds the schedule for the new budget rather than
continuing a finished one (look at the `lr` column of the resumed epoch).

In [ ]:
seed_everything(SEED)
cfg_resume = TrainConfig(run_name="resume_demo", epochs=2, lr=2e-3, patience=0)
t_first = Trainer(SmallCNN(n_classes), cfg_resume, train_loader, val_loader)
t_first.fit()

print("\n-- pretend the process died here; later we come back and resume with a larger budget --\n")
seed_everything(SEED + 1)                                # a different seed: the resumed state must come from the checkpoint
t_resumed = Trainer(SmallCNN(n_classes), replace(cfg_resume, epochs=3), train_loader, val_loader)
t_resumed.fit(resume=True)
print("history rows after resume:", [r["epoch"] for r in t_resumed.history])

# for deployment or analysis, load the best checkpoint's weights only
t_resumed.load_checkpoint(t_resumed.run_dir / "best.pt", model_only=True)
print("best checkpoint re-evaluated:", {k: round(v, 4) for k, v in t_resumed.evaluate().items()})

The resumed trainer reports epoch 3 only, its history contains all three rows, and re-evaluating the
best checkpoint reproduces the best validation accuracy that was logged. Note the deliberate use of a
different seed before resuming: the resumed model and optimizer come from the checkpoint file, not
from the freshly constructed (and immediately overwritten) `SmallCNN`.

### 2.5 Error analysis with the trained model

The confusion matrix tells us *which* classes are confused; the most confident mistakes tell us
*why*. Both take a few lines with the `predict` method.

In [ ]:
logits, targets = t_cnn.predict(val_loader)
probs = logits.softmax(1)
preds = probs.argmax(1)
cm = confusion_matrix(targets.cpu().numpy(), preds.cpu().numpy(), labels=list(range(n_classes)))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
ax.set_xticklabels([c[:7] for c in class_names], rotation=45, ha="right", fontsize=8)
ax.set_yticklabels([c[:7] for c in class_names], fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix (validation)")
ax.grid(False); plt.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()

per_class = cm.diagonal() / cm.sum(1).clip(min=1)
worst = np.argsort(per_class)[:3]
print("hardest classes:", [(class_names[i], round(float(per_class[i]), 3)) for i in worst])

wrong = (preds != targets).nonzero().flatten()
conf_wrong = wrong[probs[wrong].max(1).values.argsort(descending=True)][:8]
fig, axes = plt.subplots(1, 8, figsize=(10, 1.9))
for ax, i in zip(axes, conf_wrong):
    ax.imshow(X_val[i, 0], cmap="gray"); ax.axis("off")
    ax.set_title(f"{class_names[targets[i]][:6]}\n-> {class_names[preds[i]][:6]} ({probs[i].max():.2f})", fontsize=7)
plt.suptitle("most confident mistakes (true -> predicted)", fontsize=9); plt.tight_layout(); plt.show()

On FashionMNIST the confusions are the ones a human would make (shirt versus T-shirt versus pullover,
sneaker versus ankle boot), and the most confident errors are often ambiguous or mislabelled images.
That observation has a direct consequence for the project: the remaining error is partly *irreducible*
at this image resolution, so a bigger model is not where the next gain is. This is what error analysis
is for.

### 2.6 The reporting template

Every project ends in a report, and every report answers the same questions. The template below is the
markdown skeleton used in Section 4; copy it into your project's `REPORT.md` and fill it in as you go
rather than at the end.

```markdown
# <Project title>
**Author** · **Date** · **Code**: <repo/notebook> · **Run logs**: data/runs/<names>

## Abstract
Three to five sentences: the task, the data, the method, the main result with its uncertainty,
the main limitation.

## 1. Problem and data
Decision supported; primary and secondary metrics with target values; data source, license,
sizes, split strategy; audit findings (balance, duplicates, label noise).

## 2. Baselines
Trivial baseline, simple model, prior work; numbers on the validation set.

## 3. Method
Architecture (with parameter count), loss, optimizer and schedule, augmentation, training budget,
hardware; a table of hyperparameters; what was tuned and how.

## 4. Experiments
Main results table (mean ± std over N seeds); ablations; learning curves; compute cost.

## 5. Analysis
Error analysis (confusion matrix, hardest classes, most confident mistakes); calibration;
robustness checks; what the model learned (attribution or probes if applicable).

## 6. Limitations and negative results
What did not work and why you think so; known failure modes; what the numbers do not show.

## 7. Reproducibility
Seeds, versions (environment report), commands to reproduce each table row, data checksums.

## References
```

Notebook 17 - part 2 of 3 (project briefs (a)-(g) with starter code)

## 3. Project briefs

Each brief below is a complete specification: a goal, freely available data with approximate sizes,
milestones that build on each other, stretch goals, a rubric, and a starter cell that runs in seconds
under `FAST_RUN` on the data already in memory (or synthetic data) and that you are meant to *replace*
piece by piece with your own dataset and models. The briefs assume roughly 20–40 hours of work; the
milestones are ordered so that stopping after any of them still yields a reportable result. Pick one
project; if you are unsure, (a) and (e) have the fastest path to a strong result, (b) and (c) give the
most insight into modelling choices, (d) and (f) are the most open-ended.

The rubric is the same for every project and is applied per milestone:

| Criterion | Weight | What "excellent" looks like |
|---|---|---|
| Framing and data | 20% | Clear decision/metric, audited data, valid split, documented license |
| Baselines and experiments | 30% | Trivial and strong baselines, one-change-at-a-time ablations, several seeds |
| Analysis | 20% | Error analysis that explains the numbers and motivates the next step |
| Reproducibility | 15% | Run folders with configs/logs/checkpoints, environment report, re-runnable commands |
| Report | 15% | Follows the template; honest limitations and negative results; plots that communicate |

### 3.1 Project (a): Image classification on a custom dataset with transfer learning

**Goal.** Build a classifier for a dataset that is *not* one of the standard benchmarks, using a
pretrained backbone from torchvision, and quantify how much the pretraining helps compared with
training from scratch at your data size.

**Data (choose one).** Oxford-IIIT Pets (37 breeds, 7.4k images, ~800 MB; `torchvision.datasets.OxfordIIITPet`),
Flowers-102 (102 classes, 8k images, ~330 MB; `torchvision.datasets.Flowers102`), Food-101 (101 classes,
101k images, ~5 GB), CIFAR-100 (100 classes, 60k tiny images, ~160 MB), or your own photos with at
least 100 images per class (label them in folders and use `torchvision.datasets.ImageFolder`). Note
the license of whichever you pick.

**Milestones.**
1. Data audit (class balance, image sizes, duplicates), a fixed train/val/test split, and the
   majority-class and nearest-centroid-on-features baselines.
2. Linear probe: frozen ResNet-18/ResNet-50 features + a linear head (keep the backbone in `eval()`
   mode so that BatchNorm statistics stay frozen). Report accuracy and per-class accuracy.
3. Full fine-tuning with discriminative learning rates (Notebook 07 · Transfer Learning and Modern Vision) and augmentation
   (`transforms.v2`); compare with milestone 2 and with a small CNN trained from scratch.
4. Error analysis (confusion matrix, most confident mistakes), Grad-CAM on typical errors, and a
   learning curve: accuracy versus number of training images per class (10, 50, 100, all).
5. Report and model card.

**Stretch goals.** Swap in a ConvNeXt or ViT backbone and compare; test-time augmentation; calibrate
the classifier with temperature scaling (Notebook 16 · Interpretability, Robustness and Uncertainty);
export to ONNX and measure latency (Notebook 15 · Deployment and Model Optimization).

**Starter.** The cell below assembles a frozen ResNet-18 with a new head, adapts our 28×28 demo
images to the backbone's input (3 channels, ImageNet normalization, upsampled) and trains the head
with the `Trainer`. Under `FAST_RUN` the ImageNet weights are used only if they are already cached in
`data/torch_hub`; otherwise the backbone is randomly initialized so that the smoke test needs no
download. Replace `X_train`/`y_train` by an `ImageFolder`-backed DataLoader for your own images.

In [ ]:
class ToBackboneInput(nn.Module):
    """Upsample 1xHxW (or 3xHxW) images to 3xSxS and apply the ImageNet normalization."""

    def __init__(self, size=64):
        super().__init__()
        self.size = size
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)
        x = F.interpolate(x, size=(self.size, self.size), mode="bilinear", align_corners=False)
        return (x - self.mean) / self.std


def build_transfer_model(num_classes, imagenet_weights=True, freeze_backbone=True, size=64):
    from torchvision.models import resnet18, ResNet18_Weights
    backbone = None
    if imagenet_weights:
        try:
            backbone = resnet18(weights=ResNet18_Weights.DEFAULT)      # downloads into DATA_DIR/torch_hub
            print("resnet18: ImageNet weights loaded")
        except Exception as e:
            print(f"resnet18: ImageNet weights unavailable ({type(e).__name__}) -> random initialization")
    if backbone is None:
        backbone = resnet18(weights=None)
    if freeze_backbone:
        for p in backbone.parameters():
            p.requires_grad_(False)
    backbone.fc = nn.Linear(backbone.fc.in_features, num_classes)      # the new, trainable head
    return nn.Sequential(ToBackboneInput(size), backbone)


weights_cached = (Path(torch.hub.get_dir()) / "checkpoints" / "resnet18-f37072fd.pth").exists()
seed_everything(SEED)
model_a = build_transfer_model(n_classes, imagenet_weights=(not FAST_RUN) or weights_cached)
print(f"trainable {count_params(model_a, trainable_only=True):,} of {count_params(model_a):,} parameters")
tr_a, va_a = make_loaders(X_train, y_train, X_val, y_val, batch_size=32,
                          n_train=512 if FAST_RUN else 4000, n_val=256 if FAST_RUN else 1000)
cfg_a = TrainConfig(run_name="proj_a_linear_probe", epochs=1 if FAST_RUN else 5, lr=1e-3, patience=0,
                    notes="frozen resnet18 backbone, new head")
t_a = Trainer(model_a, cfg_a, tr_a, va_a)
t_a.fit()

With random backbone weights (the `FAST_RUN` path without a cached download) the linear probe learns
little, which is itself the control experiment of milestone 3: the same code with ImageNet weights
should jump far ahead, and the size of that jump on *your* data is the first result of the project.
Note the subtlety hidden in milestone 2: `Trainer` calls `model.train()`, which puts the frozen
BatchNorm layers into training mode and lets their running statistics drift; for a clean linear probe,
override `train_step` to call `self.model[1].eval()` first, or replace BatchNorm by frozen statistics.

### 3.2 Project (b): A word-level Transformer language model with generation and evaluation

**Goal.** Train a decoder-only Transformer on words rather than characters, evaluate it properly with
perplexity on held-out text, and study how model size, context length and tokenization affect quality.

**Data.** Tiny Shakespeare (1 MB, the course corpus), WikiText-2 (2M tokens, ~4 MB, Hugging Face
`wikitext`/`wikitext-2-raw-v1`) for a realistic vocabulary, WikiText-103 (100M tokens, ~500 MB) if
you have a GPU, or any public-domain book collection from Project Gutenberg. A word-level vocabulary
of 5k–30k types with an `<unk>` token is the baseline; a BPE tokenizer, as built in Notebook 10 · NLP
with Pretrained Transformers (Hugging Face), is milestone 4.

**Milestones.**
1. Tokenization and a vocabulary with a chosen cutoff; report the out-of-vocabulary rate on the
   validation split and the unigram-model perplexity as the trivial baseline.
2. An LSTM language model (Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs) and the Transformer
   below, both at ~1M parameters; compare validation perplexity at equal training tokens, with
   learning curves.
3. Scaling study: three model sizes and two context lengths; plot perplexity against parameters
   and note the diminishing returns (Notebook 09 · Attention and Transformers, Section 8).
4. Replace the word vocabulary by BPE and compare perplexity *per character* (perplexities over
   different vocabularies are not comparable directly).
5. Generation study: temperature, top-k and top-p; a small human evaluation of fluency; a report.

**Stretch goals.** Add a KV-cache and measure generation speed; fine-tune the model on a second
corpus and measure forgetting; implement RoPE and compare with learned positions.

**Starter.** A word tokenizer, a compact Transformer LM built from `nn.TransformerEncoderLayer` with
a causal mask and tied embeddings, a training loop with perplexity tracking, and sampling.

In [ ]:
TINY_SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
FALLBACK_TEXT = """First Citizen: Before we proceed any further, hear me speak.
All: Speak, speak.
First Citizen: You are all resolved rather to die than to famish?
All: Resolved. resolved.
First Citizen: First, you know Caius Marcius is chief enemy to the people.
All: We know't, we know't.
First Citizen: Let us kill him, and we'll have corn at our own price. Is't a verdict?
All: No more talking on't; let it be done: away, away!
Second Citizen: One word, good citizens.
First Citizen: We are accounted poor citizens, the patricians good. What authority surfeits on would
relieve us: if they would yield us but the superfluity, while it were wholesome, we might guess they
relieved us humanely; but they think we are too dear: the leanness that afflicts us, the object of our
misery, is as an inventory to particularise their abundance; our sufferance is a gain to them.
"""


def load_text_corpus(path=DATA_DIR / "tinyshakespeare.txt"):
    if path.exists():
        return path.read_text(encoding="utf-8"), "cached"
    try:
        import urllib.request
        with urllib.request.urlopen(TINY_SHAKESPEARE_URL, timeout=20) as r:
            text = r.read().decode("utf-8")
        path.write_text(text, encoding="utf-8")
        return text, "downloaded"
    except Exception as e:
        print(f"download failed ({type(e).__name__}) -> built-in fallback text")
        return FALLBACK_TEXT * 20, "fallback"


class WordTokenizer:
    def __init__(self, text, max_vocab=8000):
        self.words = re.findall(r"[a-z']+|[.,;:!?]", text.lower())
        counts = Counter(self.words)
        self.itos = ["<unk>"] + [w for w, _ in counts.most_common(max_vocab - 1)]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, words):
        return torch.tensor([self.stoi.get(w, 0) for w in words], dtype=torch.long)

    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids)


class WordTransformerLM(nn.Module):
    """Decoder-only Transformer LM from nn.TransformerEncoderLayer + causal mask, tied embeddings."""

    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2, context=64, dropout=0.1):
        super().__init__()
        self.context = context
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(context, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, 4 * d_model, dropout, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight                          # weight tying
        nn.init.normal_(self.tok_emb.weight, std=0.02)                  # GPT-2 style: small embeddings keep the
        nn.init.normal_(self.pos_emb.weight, std=0.02)                  # tied logits small at initialization

    def forward(self, idx):
        T = idx.size(1)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        return self.head(self.ln_f(self.blocks(x, mask=mask, is_causal=True)))

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        self.eval()
        for _ in range(max_new_tokens):
            logits = self(idx[:, -self.context:])[:, -1] / temperature
            idx = torch.cat([idx, torch.multinomial(logits.softmax(-1), 1)], dim=1)
        return idx


def get_lm_batch(data, context, batch_size):
    ix = torch.randint(len(data) - context - 1, (batch_size,))
    x = torch.stack([data[i:i + context] for i in ix]); y = torch.stack([data[i + 1:i + context + 1] for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def lm_perplexity(model, data, context, n_batches=10):
    model.eval()
    losses = [F.cross_entropy(model(x).flatten(0, 1), y.flatten()).item()
              for x, y in (get_lm_batch(data, context, 32) for _ in range(n_batches))]
    return math.exp(float(np.mean(losses)))


text_b, src_b = load_text_corpus()
tok_b = WordTokenizer(text_b, max_vocab=4000 if FAST_RUN else 8000)
data_b = tok_b.encode(tok_b.words)
n_b = int(0.9 * len(data_b))
train_b, val_b = data_b[:n_b], data_b[n_b:]
oov = (val_b == 0).float().mean().item()
print(f"corpus ({src_b}): {len(data_b):,} words, vocab {len(tok_b.itos):,}, validation OOV rate {oov:.1%}")

CONTEXT_B = 32 if FAST_RUN else 64
STEPS_B = 80 if FAST_RUN else 1500
seed_everything(SEED)
lm = WordTransformerLM(len(tok_b.itos), d_model=128, n_layers=2, context=CONTEXT_B).to(device)
opt_b = torch.optim.AdamW(lm.parameters(), lr=1e-3, weight_decay=0.01)
print(f"model: {count_params(lm) / 1e6:.2f}M parameters | unigram baseline perplexity (train-set entropy) "
      f"{math.exp(-sum(c / len(train_b) * math.log(c / len(train_b)) for c in Counter(train_b.tolist()).values())):.0f}")
t0 = time.time()
for step in range(1, STEPS_B + 1):
    lm.train()
    xb, yb = get_lm_batch(train_b, CONTEXT_B, 32)
    loss = F.cross_entropy(lm(xb).flatten(0, 1), yb.flatten())
    opt_b.zero_grad(set_to_none=True); loss.backward()
    nn.utils.clip_grad_norm_(lm.parameters(), 1.0); opt_b.step()
    if step % max(1, STEPS_B // 4) == 0:
        print(f"step {step:5d} | train loss {loss.item():.3f} | val perplexity {lm_perplexity(lm, val_b, CONTEXT_B):.1f} | {time.time() - t0:.0f}s")
prompt_b = tok_b.encode(["the", "king"]).unsqueeze(0).to(device)
print("sample:", tok_b.decode(lm.generate(prompt_b, 30, temperature=0.8)[0].tolist()))

Eighty steps are enough to see the perplexity fall from the vocabulary size towards the unigram
baseline; the full budget gets a 2-layer model to a perplexity well below it and to samples with
Shakespearean phrase structure. Two measurement rules from milestone 1 are already visible in the
output: report the out-of-vocabulary rate, because every `<unk>` is an easy prediction that flatters
perplexity, and always compare against the unigram baseline, which is surprisingly strong on small
corpora.

### 3.3 Project (c): Time-series forecasting with sequence models versus Transformers

**Goal.** Forecast a multivariate or univariate series several steps ahead, and answer with evidence
the question that the literature keeps debating: do attention-based models beat recurrent models and
simple baselines on your data?

**Data.** ETT (Electricity Transformer Temperature; ETTh1 is a 17k-row CSV of ~2.5 MB, hourly, 7
variables), the Jena Climate dataset (420k rows at 10-minute resolution, ~13 MB zipped, 14 variables),
the UCI Electricity Load Diagrams (370 clients at 15-minute resolution, ~250 MB zipped), M4 (100k
series, ~100 MB), or a series you care
about (energy meter, web traffic, weather station). Split *chronologically*: the validation and test
periods must come after the training period.

**Milestones.**
1. Audit: plot the series, identify seasonalities (autocorrelation), missing values, regime changes;
   fit the scaler on the training period only.
2. Baselines: naive (last value), seasonal naive (value one period ago), and a linear model on the
   window; report MAE and MASE (Hyndman and Koehler, 2006: MAE scaled by the MAE of the one-step
   seasonal-naive forecast on the training period) at horizons 1, 12 and 24.
3. GRU/LSTM forecaster and a Transformer-encoder forecaster (below), tuned with a small random
   search; error versus horizon plot; several seeds.
4. Ablations that matter for time series: context length, direct multi-output versus
   autoregressive decoding, with and without positional encoding, input normalization per window
   (RevIN-style) versus global.
5. Report, including a discussion of why (or whether) the simple baselines were hard to beat.

**Stretch goals.** Probabilistic forecasts (predict quantiles with the pinball loss and evaluate
coverage); multivariate inputs with a channel-independent versus channel-mixing comparison;
a patching Transformer (PatchTST-style).

**Starter.** A synthetic series with trend, two seasonalities and a regime component; windowing into
supervised examples; naive and seasonal-naive baselines; a GRU and a Transformer forecaster trained
with the `Trainer` using `F.mse_loss` and an MAE metric in the original units.

In [ ]:
def make_series(n, seed=SEED):
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = (0.001 * t + 2.0 * np.sin(2 * np.pi * t / 24) + 0.8 * np.sin(2 * np.pi * t / 168)
         + 0.5 * ((t // 500) % 2) + 0.3 * rng.standard_normal(n))
    return y.astype(np.float32)


def make_windows(series, context, horizon):
    n = len(series) - context - horizon + 1
    X = np.stack([series[i:i + context] for i in range(n)])
    Y = np.stack([series[i + context:i + context + horizon] for i in range(n)])
    return torch.tensor(X).unsqueeze(-1), torch.tensor(Y)           # (N, context, 1), (N, horizon)


class GRUForecaster(nn.Module):
    def __init__(self, horizon, hidden=64):
        super().__init__()
        self.gru = nn.GRU(1, hidden, batch_first=True)
        self.head = nn.Linear(hidden, horizon)

    def forward(self, x):
        _, h = self.gru(x)
        return self.head(h[-1])


class TransformerForecaster(nn.Module):
    def __init__(self, context, horizon, d_model=64, n_heads=4, n_layers=2):
        super().__init__()
        self.proj = nn.Linear(1, d_model)
        self.pos = nn.Embedding(context, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, 4 * d_model, 0.1, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        h = self.proj(x) + self.pos(torch.arange(x.size(1), device=x.device))
        return self.head(self.enc(h)[:, -1])


CONTEXT_C, HORIZON_C, SEASON_C = 48, 12, 24
series = make_series(3000 if FAST_RUN else 8000)
n_split = int(0.8 * len(series))
mean_c, std_c = float(series[:n_split].mean()), float(series[:n_split].std())     # scaler fitted on train only
series_n = (series - mean_c) / std_c
Xc_tr, Yc_tr = make_windows(series_n[:n_split], CONTEXT_C, HORIZON_C)
Xc_va, Yc_va = make_windows(series_n[n_split:], CONTEXT_C, HORIZON_C)


def mae_metric(outputs, targets):
    return {"mae": (outputs - targets).abs().mean().item() * std_c}    # back in original units


naive = Xc_va[:, -1, 0:1].expand(-1, HORIZON_C)
seasonal = Xc_va[:, -SEASON_C:-SEASON_C + HORIZON_C, 0]
forecast_results = {"naive (last value)": mae_metric(naive, Yc_va)["mae"],
                    "seasonal naive": mae_metric(seasonal, Yc_va)["mae"]}

tr_c, va_c = make_loaders(Xc_tr, Yc_tr, Xc_va, Yc_va, batch_size=64)
EPOCHS_C = 4 if FAST_RUN else 15
for name, model_c in [("GRU", GRUForecaster(HORIZON_C)), ("Transformer", TransformerForecaster(CONTEXT_C, HORIZON_C))]:
    seed_everything(SEED)
    cfg_c = TrainConfig(run_name=f"proj_c_{name.lower()}", epochs=EPOCHS_C, lr=2e-3, monitor="val_mae", mode="min", patience=4)
    t_c = Trainer(model_c, cfg_c, tr_c, va_c, loss_fn=F.mse_loss, metric_fn=mae_metric, verbose=False)
    t_c.fit()
    forecast_results[name] = t_c.best_score
print(f"\n{'model':22s} {'val MAE':>8s} {'rel. MAE':>9s}")
for name, mae in forecast_results.items():
    print(f"{name:22s} {mae:8.3f} {mae / forecast_results['seasonal naive']:9.2f}")

The seasonal-naive baseline is strong on a series with clean periodicity, which is why the ratio of a
model's MAE to the seasonal-naive MAE (the last column; MASE is the same idea with the scale computed
on the training period) is the number to report: values below 1 mean the model has learned something
beyond "same as yesterday". Under `FAST_RUN` four epochs may leave the learned models above the
baseline; the full run puts both well below it, and which of the two wins depends on the seed as much
as on the architecture, so milestone 3 requires several seeds before claiming anything.

### 3.4 Project (d): A conditional diffusion model on a custom image set

**Goal.** Train a class-conditional DDPM with classifier-free guidance on an image collection of your
choice, evaluate sample quality and diversity, and study the effect of the guidance scale and the
number of sampling steps.

**Data.** FashionMNIST or MNIST (already downloaded; 28×28 grayscale, a safe first target), KMNIST
(70k 28×28 grayscale Japanese characters, ~20 MB, CC BY-SA 4.0; `torchvision.datasets.KMNIST`),
CIFAR-10 (32×32 colour, 170 MB; recognizable samples need a few GPU-hours), or your own collection of
≥5k images downsampled to 32×32
or 64×64 (icons, textures, satellite tiles). Grayscale at 16–32 px is where a laptop GPU is
sufficient.

**Milestones.**
1. Data pipeline scaled to $[-1, 1]$, the forward noising process visualized (Notebook 12 · GANs and
   Diffusion Models), and the
   noise schedule chosen with $\bar\alpha_T \approx 0$ verified numerically.
2. Unconditional DDPM with the U-Net below; loss curve; samples every N steps; a nearest-neighbour
   check that samples are not copies of training images.
3. Class-conditional training with label dropout and classifier-free guidance at sampling time;
   guidance-scale sweep with a grid of samples.
4. Evaluation: train a small classifier on real data and report (i) its accuracy on generated
   images of each class as a "recognizability" score and (ii) a feature-space distance between real
   and generated images (an FID proxy, Notebook 11 · Autoencoders and Variational Autoencoders); DDIM
   sampling with 10/50/200 steps versus
   quality.
5. Report with sample grids and a discussion of failure modes (mode collapse does not happen, but
   blurriness and class leakage do).

**Stretch goals.** Add attention at the lowest resolution; an EMA of the weights for sampling;
a latent-diffusion variant using the VAE of Notebook 11; flow matching instead of DDPM.

**Starter.** A compact class-conditional U-Net with sinusoidal time embeddings and a `DDPM` helper
implementing the forward process, the $\epsilon$-prediction loss with label dropout, and ancestral
sampling with classifier-free guidance. It trains for a few hundred steps on 16×16 versions of the demo
images; under `FAST_RUN` the samples are blurry blobs with the right coarse shape, which is the
expected state after 150 steps.

In [ ]:
IMG_D = 16


def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=1)


class ResBlock(nn.Module):
    def __init__(self, cin, cout, emb_dim):
        super().__init__()
        self.norm1, self.conv1 = nn.GroupNorm(8, cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.emb = nn.Linear(emb_dim, cout)
        self.norm2, self.conv2 = nn.GroupNorm(8, cout), nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, e):
        h = self.conv1(F.silu(self.norm1(x))) + self.emb(e)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class CondUNet(nn.Module):
    """16x16 -> 8x8 -> 4x4 -> 8x8 -> 16x16 with time and class conditioning (null class for CFG)."""

    def __init__(self, n_classes, ch=32, emb_dim=128):
        super().__init__()
        self.emb_dim, self.null_class = emb_dim, n_classes
        self.time_mlp = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.SiLU(), nn.Linear(emb_dim, emb_dim))
        self.class_emb = nn.Embedding(n_classes + 1, emb_dim)
        self.inc = nn.Conv2d(1, ch, 3, padding=1)
        self.down1, self.down2 = ResBlock(ch, 2 * ch, emb_dim), ResBlock(2 * ch, 4 * ch, emb_dim)
        self.mid = ResBlock(4 * ch, 4 * ch, emb_dim)
        self.up1, self.up2 = ResBlock(6 * ch, 2 * ch, emb_dim), ResBlock(3 * ch, ch, emb_dim)
        self.out = nn.Conv2d(ch, 1, 3, padding=1)

    def forward(self, x, t, y):
        e = self.time_mlp(sinusoidal_embedding(t, self.emb_dim)) + self.class_emb(y)
        h0 = self.inc(x)
        h1 = self.down1(F.avg_pool2d(h0, 2), e)
        h2 = self.down2(F.avg_pool2d(h1, 2), e)
        h = self.mid(h2, e)
        h = self.up1(torch.cat([F.interpolate(h, scale_factor=2.0, mode="nearest"), h1], 1), e)
        h = self.up2(torch.cat([F.interpolate(h, scale_factor=2.0, mode="nearest"), h0], 1), e)
        return self.out(F.silu(h))


class DDPM:
    def __init__(self, T, device):
        self.T = T
        self.betas = torch.linspace(1e-4, 20.0 / T, T, device=device)      # sum(betas) ~ 10 -> alpha_bar_T ~ 0
        self.alphas = 1.0 - self.betas
        self.abar = torch.cumprod(self.alphas, 0)

    def loss(self, model, x0, y, p_uncond=0.1):
        t = torch.randint(0, self.T, (x0.size(0),), device=x0.device)
        noise = torch.randn_like(x0)
        ab = self.abar[t][:, None, None, None]
        xt = ab.sqrt() * x0 + (1 - ab).sqrt() * noise
        y = torch.where(torch.rand(y.shape, device=y.device) < p_uncond, torch.full_like(y, model.null_class), y)
        return F.mse_loss(model(xt, t, y), noise)

    @torch.no_grad()
    def sample(self, model, y, guidance=2.0, shape=(1, IMG_D, IMG_D)):
        model.eval()
        x = torch.randn(len(y), *shape, device=y.device)
        y_null = torch.full_like(y, model.null_class)
        for t in reversed(range(self.T)):
            tt = torch.full((len(y),), t, device=y.device)
            eps = model(x, tt, y_null) + guidance * (model(x, tt, y) - model(x, tt, y_null))
            mean = (x - self.betas[t] / (1 - self.abar[t]).sqrt() * eps) / self.alphas[t].sqrt()
            x = mean + self.betas[t].sqrt() * torch.randn_like(x) if t > 0 else mean
        return x.clamp(-1, 1)


T_D = 100 if FAST_RUN else 400
STEPS_D = 150 if FAST_RUN else 3000
seed_everything(SEED)
unet = CondUNet(n_classes).to(device)
ddpm = DDPM(T_D, device)
opt_d = torch.optim.AdamW(unet.parameters(), lr=2e-3 if FAST_RUN else 5e-4)
X_d = (F.interpolate(X_train, size=(IMG_D, IMG_D), mode="bilinear", align_corners=False) * 2 - 1)
print(f"U-Net {count_params(unet) / 1e6:.2f}M parameters | alpha_bar_T = {ddpm.abar[-1].item():.2e}")
t0 = time.time()
for step in range(1, STEPS_D + 1):
    unet.train()
    idx = torch.randint(len(X_d), (64,))
    loss = ddpm.loss(unet, X_d[idx].to(device), y_train[idx].to(device))
    opt_d.zero_grad(set_to_none=True); loss.backward(); opt_d.step()
    if step % max(1, STEPS_D // 4) == 0:
        print(f"step {step:5d} | loss {loss.item():.4f} | {time.time() - t0:.0f}s")

show_classes = [0, 1] if n_classes < 4 else [0, 3]
y_s = torch.tensor([c for c in show_classes for _ in range(8)], device=device)
samples = ddpm.sample(unet, y_s, guidance=2.0).cpu()
fig, axes = plt.subplots(len(show_classes), 8, figsize=(8, 2.2))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i, 0], cmap="gray", vmin=-1, vmax=1); ax.axis("off")
    if i % 8 == 0:
        ax.set_title(class_names[show_classes[i // 8]][:10], fontsize=8, loc="left")
plt.suptitle(f"class-conditional samples after {STEPS_D} steps (guidance 2.0)", fontsize=9)
plt.tight_layout(); plt.show()

The loss curve of a diffusion model is a poor progress indicator (it plateaus early while sample
quality keeps improving), which is why milestone 2 asks for periodic sample grids and milestone 4
for a classifier-based recognizability score. The `guidance` argument is the first knob to sweep:
at 0 the samples ignore the class, at 1 they follow the conditional model, above 1 they become more
prototypical and less diverse.

### 3.5 Project (e): Fine-tuning a pretrained Transformer for a text task with LoRA and a proper evaluation

**Goal.** Adapt a pretrained encoder (or a small decoder) to a text classification or
token-classification task with LoRA, and produce an evaluation that would survive peer review:
confidence intervals, several seeds, a comparison with full fine-tuning and with a non-neural
baseline, and an error analysis.

**Data.** SST-2 or another GLUE task (~7 MB via `datasets`), IMDB reviews (25k/25k, ~80 MB), AG News
(120k, 4 classes, ~30 MB), TweetEval emotion (~5k tweets, small and noisy, good for studying variance),
CoNLL-2003 for named-entity recognition (token classification, ~1 MB; its Reuters text is licensed for
research use only), or a labelled corpus from your own domain. Check the license and whether the test
labels are public.

**Milestones.**
1. Data audit (length distribution, class balance, duplicates, label noise on a hand-checked sample),
   a TF-IDF + logistic-regression baseline (scikit-learn), and the majority baseline.
2. Full fine-tuning of DistilBERT (Notebook 10 · NLP with Pretrained Transformers (Hugging Face))
   with a hand-written loop; three seeds; report
   mean ± std and a bootstrap confidence interval of the test accuracy/F1.
3. LoRA (from scratch or `peft`) at ranks 2, 8, 32 on attention only versus all linear layers;
   trainable parameters versus accuracy plot; wall-clock and memory comparison.
4. Error analysis: hardest examples, performance by input length, calibration (Notebook 16 ·
   Interpretability, Robustness and Uncertainty); a paired comparison (same test items) between full
   fine-tuning and the best LoRA run.
5. Report and model card, including the base model's license and the data's license.

**Stretch goals.** QLoRA on a 0.5–1B decoder with a classification head; label-efficiency curve
(accuracy versus number of training examples); distil the fine-tuned model into a small student
(Notebook 15 · Deployment and Model Optimization).

**Starter.** A self-contained `LoRALinear` with merging, an `apply_lora` helper, and two evaluation
utilities that most student reports lack: a bootstrap confidence interval for accuracy and a paired
bootstrap for the *difference* between two models evaluated on the same items. The demo applies LoRA
to DistilBERT if its weights are already cached (or `FAST_RUN` is off), and otherwise to a small
in-notebook Transformer classifier, then runs the statistics on synthetic predictions.

In [ ]:
class LoRALinear(nn.Module):
    """Frozen nn.Linear + trainable low-rank update (alpha/r) * B A, mergeable into the base weight."""

    def __init__(self, base, r=8, alpha=16.0):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.merged = False

    def forward(self, x):
        out = self.base(x)
        return out if self.merged else out + self.scaling * ((x @ self.lora_A.t()) @ self.lora_B.t())

    @torch.no_grad()
    def merge(self):
        if not self.merged:
            self.base.weight += self.scaling * (self.lora_B @ self.lora_A); self.merged = True


def apply_lora(model, targets, r=8, alpha=16.0):
    n = 0
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear) and name.split(".")[-1] in targets:
            parent = model.get_submodule(name.rsplit(".", 1)[0]) if "." in name else model
            setattr(parent, name.split(".")[-1], LoRALinear(module, r, alpha)); n += 1
    for name, p in model.named_parameters():
        p.requires_grad_("lora_A" in name or "lora_B" in name or name.startswith("classifier") or name.startswith("pre_classifier"))
    return n


def bootstrap_ci(correct, n_boot=2000, seed=SEED):
    """95% bootstrap CI of the accuracy given a 0/1 array of per-item correctness."""
    rng = np.random.default_rng(seed)
    correct = np.asarray(correct, dtype=float)
    boots = rng.choice(correct, size=(n_boot, len(correct)), replace=True).mean(1)
    return correct.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)


def paired_bootstrap_diff(correct_a, correct_b, n_boot=2000, seed=SEED):
    """95% CI of accuracy(A) - accuracy(B) when both were evaluated on the same items."""
    rng = np.random.default_rng(seed)
    diff = np.asarray(correct_a, dtype=float) - np.asarray(correct_b, dtype=float)
    boots = rng.choice(diff, size=(n_boot, len(diff)), replace=True).mean(1)
    return diff.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)


class TinyTextClassifier(nn.Module):
    """In-notebook stand-in for a pretrained encoder (same module naming pattern as DistilBERT's FFN)."""

    def __init__(self, vocab=5000, d=128, n_layers=2, n_classes=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        layer = nn.TransformerEncoderLayer(d, 4, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.pre_classifier, self.classifier = nn.Linear(d, d), nn.Linear(d, n_classes)

    def forward(self, ids):
        return self.classifier(F.relu(self.pre_classifier(self.encoder(self.emb(ids))[:, 0])))


hf_cache = DATA_DIR / "hf_cache"
distilbert_cached = (hf_cache / "models--distilbert-base-uncased").exists()
model_e, targets_e = None, None
if HF_AVAILABLE and (distilbert_cached or not FAST_RUN):
    try:
        model_e = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2,
                                                                     cache_dir=str(hf_cache))
        targets_e = ("q_lin", "v_lin")
        print("using DistilBERT")
    except Exception as e:
        print(f"could not load DistilBERT ({type(e).__name__}) -> using the in-notebook classifier")
if model_e is None:
    model_e, targets_e = TinyTextClassifier(), ("linear1", "linear2")     # the FFN linears of each block
    print("using TinyTextClassifier")
before = count_params(model_e)
n_replaced = apply_lora(model_e, targets_e, r=8)
print(f"LoRA applied to {n_replaced} layers: trainable {count_params(model_e, trainable_only=True):,} "
      f"of {count_params(model_e):,} (base model had {before:,})")

# evaluation statistics on synthetic per-item correctness (replace with your models' predictions)
rng_e = np.random.default_rng(SEED)
n_items = 500
correct_full = rng_e.random(n_items) < 0.89
# LoRA on the same items, with errors correlated with full fine-tuning: it loses 5% of the items that
# full fine-tuning gets right and recovers 10% of the ones it gets wrong
flip = np.where(correct_full, rng_e.random(n_items) < 0.05, rng_e.random(n_items) < 0.10)
correct_lora = correct_full ^ flip
for name, c in [("full fine-tuning", correct_full), ("LoRA", correct_lora)]:
    m, lo, hi = bootstrap_ci(c)
    print(f"{name:18s} accuracy {m:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  (half-width {(hi - lo) / 2:.3f})")
d, lo, hi = paired_bootstrap_diff(correct_full, correct_lora)
print(f"paired difference (full - LoRA): {d:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]"
      + ("  -> not significant" if lo < 0 < hi else "  -> significant"))
# the same difference if the two accuracies were (wrongly) treated as independent samples
se_indep = math.sqrt(correct_full.var() / n_items + correct_lora.var() / n_items)
print(f"unpaired normal approximation:   {d:+.3f}  95% CI [{d - 1.96 * se_indep:+.3f}, {d + 1.96 * se_indep:+.3f}]"
      f"  -> the paired interval is {(hi - lo) / (2 * 1.96 * se_indep):.2f}x as wide")
model_e = None

The three intervals illustrate why milestone 4 asks for a *paired* comparison. With 500 test items
each marginal interval is about ±3 points wide, so the two overlap or nearly do although the
underlying accuracies differ by about three points. Treating the two accuracies as independent gives
an interval for the difference that is roughly twice as wide as the paired one: the two models' errors
are strongly correlated (they fail on the same hard items), and pairing removes that shared variance.
Report the paired interval, never the overlap of the marginal ones.

### 3.6 Project (f): A reinforcement-learning agent with a method comparison

**Goal.** Solve a Gymnasium classic-control or toy-text environment with at least three methods from
Notebook 13 · Reinforcement Learning (tabular Q-learning or SARSA where the state space allows,
REINFORCE, DQN, A2C/PPO), and
compare them on sample efficiency, final performance and variance across seeds.

**Data.** None to download; environments are simulators. Toy text: `FrozenLake-v1` (slippery and not),
`Taxi-v3`, `CliffWalking-v0`. Classic control: `CartPole-v1`, `Acrobot-v1`, `MountainCar-v0` (sparse
reward, a good stretch target), `LunarLander-v3` needs the `box2d` extra and is optional.

**Milestones.**
1. A random-agent baseline and a hand-written heuristic policy for the chosen environment, evaluated
   over 100 episodes with mean and standard deviation; a fixed evaluation protocol (greedy policy,
   fixed seeds, fixed episode cap).
2. A tabular method on the toy-text environment (Q-learning below) or DQN on the control task;
   learning curves averaged over 5 seeds with a shaded standard-deviation band.
3. Two further methods (policy gradient, actor–critic); a comparison table at equal environment
   steps; a hyperparameter sensitivity study for the learning rate and the exploration schedule.
4. Analysis: visualize the learned policy/value function (grid heatmaps for toy text; state-space
   plots for control); failure modes (divergence, forgetting, reward hacking on shaped rewards).
5. Report with a discussion of variance: RL results are notoriously seed-dependent
   (Henderson et al., 2018), and your report should show it rather than hide it.

**Stretch goals.** Double or dueling DQN; PPO with vectorized environments; transfer of a policy
between the slippery and deterministic lake; a small reward-shaping study with its pitfalls.

**Starter.** An environment factory with a built-in gridworld fallback that mimics the Gymnasium API,
tabular Q-learning with a linear epsilon schedule, an evaluation protocol, and the learning-curve plot.

In [ ]:
class MiniFrozenLake:
    """Deterministic 4x4 FrozenLake clone with the Gymnasium API (used when gymnasium is missing)."""
    MAP = ["SFFF", "FHFH", "FFFH", "HFFG"]

    def __init__(self):
        self.observation_space = SimpleNamespace(n=16)
        self.action_space = SimpleNamespace(n=4, sample=lambda: random.randrange(4))
        self.s = 0

    def reset(self, seed=None):
        self.s = 0
        return self.s, {}

    def step(self, a):
        r, c = divmod(self.s, 4)
        r, c = [(r, max(c - 1, 0)), (min(r + 1, 3), c), (r, min(c + 1, 3)), (max(r - 1, 0), c)][a]
        self.s = 4 * r + c
        tile = self.MAP[r][c]
        return self.s, float(tile == "G"), tile in "HG", False, {}


def make_env():
    if GYM_AVAILABLE:
        return gym.make("FrozenLake-v1", is_slippery=False)
    return MiniFrozenLake()


def q_learning(env, episodes, alpha=0.1, gamma=0.99, eps_start=1.0, eps_end=0.05, max_steps=100, seed=SEED):
    rng = np.random.default_rng(seed)
    nS, nA = env.observation_space.n, env.action_space.n
    Q, returns = np.zeros((nS, nA)), []
    for ep in range(episodes):
        eps = max(eps_end, eps_start * (1 - ep / (0.8 * episodes)))
        s, _ = env.reset(seed=int(rng.integers(2**31 - 1)))
        total = 0.0
        for _ in range(max_steps):
            a = int(rng.integers(nA)) if rng.random() < eps else int(np.argmax(Q[s] + 1e-9 * rng.random(nA)))
            s2, r, terminated, truncated, _ = env.step(a)
            Q[s, a] += alpha * (r + gamma * (0.0 if terminated else Q[s2].max()) - Q[s, a])
            s, total = s2, total + r
            if terminated or truncated:
                break
        returns.append(total)
    return Q, np.array(returns)


def evaluate_policy(env, policy, episodes=100, max_steps=100, seed=SEED + 1):
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(episodes):
        s, _ = env.reset(seed=int(rng.integers(2**31 - 1)))
        total = 0.0
        for _ in range(max_steps):
            s, r, terminated, truncated, _ = env.step(policy(s, rng))
            total += r
            if terminated or truncated:
                break
        scores.append(total)
    return float(np.mean(scores)), float(np.std(scores))


env = make_env()
EPISODES_F = 300 if FAST_RUN else 2000
Q_table, ep_returns = q_learning(env, EPISODES_F)
rl_results = {"random": evaluate_policy(env, lambda s, rng: int(rng.integers(env.action_space.n))),
              "Q-learning (greedy)": evaluate_policy(env, lambda s, rng: int(np.argmax(Q_table[s])))}
print(f"environment: {type(env).__name__ if not GYM_AVAILABLE else env.spec.id}")
for name, (m, s) in rl_results.items():
    print(f"{name:22s} success rate {m:.2f} ± {s:.2f} over 100 evaluation episodes")
window = 20
smooth = np.convolve(ep_returns, np.ones(window) / window, mode="valid")
plt.plot(np.arange(window - 1, EPISODES_F), smooth)
plt.xlabel("episode"); plt.ylabel(f"return (moving average, {window})"); plt.title("Tabular Q-learning on FrozenLake")
plt.tight_layout(); plt.show()

On the deterministic lake, Q-learning reaches a 100% success rate within a few hundred episodes; the
slippery version (`is_slippery=True`) is the real task, where the optimal policy succeeds only about
three quarters of the time and where variance across seeds becomes the story. Milestone 2's demand for
five seeds with a shaded band is not decoration: a single learning curve in RL is anecdote.

### 3.7 Project (g), optional: reproduce a small paper result

**Goal.** Choose one quantitative claim from a paper whose experiments fit your compute (a table row,
a figure), re-implement it from the paper's description, and report whether you reproduced it, with
the same rigour you would want from the original authors.

**Suggestions that fit a laptop or a small GPU.** The degradation problem of plain deep networks
versus residual networks on CIFAR-10 (He et al., 2016, Figure 1); the effect of BatchNorm on
trainability at high learning rates (Ioffe and Szegedy, 2015); the LoRA finding that a very low rank
already matches full fine-tuning (Hu et al., 2021, Section 7.2), scaled down to DistilBERT on a small
GLUE task; the calibration gap of modern networks and its fix by temperature scaling
(Guo et al., 2017); "grokking" on modular arithmetic (Power et al., 2022); the double-descent curve
for a small CNN with label noise (Nakkiran et al., 2020).

**Milestones.**
1. A one-page summary of the claim: the exact numbers, the setup as far as it is specified, and a
   list of every detail the paper does *not* specify (these are your degrees of freedom).
2. The implementation, with a unit test against any published artefact (a released checkpoint, a
   table of intermediate values).
3. The experiment grid, run over several seeds, with results in the paper's format next to the
   paper's numbers.
4. A discussion of the discrepancies and of which unspecified details you had to guess; if you
   cannot reproduce the claim, the report should make clear what you tried.

**Rubric emphasis.** Reproducibility (25%) and analysis (25%) weigh more than in the other projects;
a well-documented failed reproduction earns full marks.

**Starter.** An experiment-grid runner built on the `Trainer` and a summary function that reports
mean ± std over seeds, applied to a small claim we can test in seconds: "BatchNorm improves the
accuracy of a small CNN at a short training budget".

In [ ]:
def run_grid(configs, seeds, n_train, n_val, batch_size=64, verbose=False):
    """configs: {name: (make_model, TrainConfig)}. Runs every (config, seed) pair; returns rows."""
    tr, va = make_loaders(X_train, y_train, X_val, y_val, batch_size, n_train=n_train, n_val=n_val)
    rows = []
    for name, (make_model, cfg) in configs.items():
        for seed in seeds:
            seed_everything(seed)
            t = Trainer(make_model(), replace(cfg, run_name=f"{cfg.run_name}_seed{seed}", seed=seed), tr, va, verbose=verbose)
            t.fit()
            rows.append({"config": name, "seed": seed, "best": t.best_score, "time": sum(r["time"] for r in t.history)})
            print(f"  {name:14s} seed {seed}: best {cfg.monitor} {t.best_score:.4f} ({rows[-1]['time']:.1f}s)")
    return rows


def summarize_grid(rows, key="best"):
    """Group rows by config; print and return {config: (mean, std, n)}."""
    out = {}
    for name in dict.fromkeys(r["config"] for r in rows):
        vals = np.array([r[key] for r in rows if r["config"] == name])
        out[name] = (vals.mean(), vals.std(ddof=1) if len(vals) > 1 else 0.0, len(vals))
    print(f"\n{'config':16s} {'mean':>8s} {'std':>7s} {'n':>3s}")
    for name, (m, s, n) in out.items():
        print(f"{name:16s} {m:8.4f} {s:7.4f} {n:3d}")
    return out


SEEDS_G = [0, 1] if FAST_RUN else [0, 1, 2, 3, 4]
cfg_g = TrainConfig(run_name="proj_g_bn", epochs=1 if FAST_RUN else 5, lr=2e-3, patience=0)
grid_rows = run_grid({"with BN": (lambda: SmallCNN(n_classes, use_bn=True), cfg_g),
                      "without BN": (lambda: SmallCNN(n_classes, use_bn=False), replace(cfg_g, run_name="proj_g_nobn"))},
                     seeds=SEEDS_G, n_train=1000 if FAST_RUN else 10000, n_val=500 if FAST_RUN else 2000)
grid_summary = summarize_grid(grid_rows)

Whether the claim "holds" depends on the budget: at one epoch on 1,000 images the difference between
the two configurations is usually smaller than the standard deviation across two seeds, and the honest
conclusion is "not resolved at this budget". That sentence, with the numbers that support it, is a
legitimate result of a reproduction, and the same grid runner produces it at any scale.

Notebook 17 - part 3 of 3 (reproducibility and reporting, what next, glossary, exercises, summary, further reading)

## 4. Reproducibility and reporting

### 4.1 The experiment log

A result you cannot trace back to a configuration, a seed and a code version is a rumour. The
`Trainer` produced, for every run, a folder with `config.json` (the complete `TrainConfig`),
`log.jsonl` (one JSON object per epoch) and two checkpoints. That folder *is* the experiment log; a
project with fifty runs has fifty such folders, and a ten-line script can turn them into the results
table of the report. Two habits make this work: never reuse a `run_name` for a different
configuration (append a version or a date), and write a one-line conclusion for every run into a
shared `EXPERIMENTS.md` immediately after looking at it ("lr 5e-3 diverged at epoch 2; keep 2e-3").
Let us look inside the folder of the CNN run from Section 2.

In [ ]:
run_dir = t_cnn.run_dir
print("files:", sorted(p.name for p in run_dir.iterdir()))
print("\nconfig.json:\n" + (run_dir / "config.json").read_text(encoding="utf-8"))
print("log.jsonl (first rows):")
for line in (run_dir / "log.jsonl").read_text(encoding="utf-8").splitlines()[:3]:
    print("  " + line[:140])

# from run folders to a results table: read every log under RUNS_DIR
rows_all = []
for cfg_path in sorted(RUNS_DIR.glob("*/config.json")):
    log_path = cfg_path.with_name("log.jsonl")
    try:                                     # skip folders written by other tools/notebooks
        cfg_d = json.loads(cfg_path.read_text(encoding="utf-8"))
        hist = [json.loads(l) for l in log_path.read_text(encoding="utf-8").splitlines() if l.strip()]
        key = cfg_d["monitor"]
        best = (max if cfg_d.get("mode", "max") == "max" else min)(r[key] for r in hist if key in r)
        rows_all.append((cfg_d["run_name"], cfg_d["seed"], cfg_d["epochs"], len(hist), key, best))
    except (OSError, KeyError, ValueError, json.JSONDecodeError):
        continue
print(f"\n{'run':28s} {'seed':>4s} {'budget':>6s} {'done':>4s} {'metric':>8s} {'best':>8s}")
for r in rows_all[:12]:
    print(f"{r[0]:28s} {r[1]:4d} {r[2]:6d} {r[3]:4d} {r[4]:>8s} {r[5]:8.4f}")
print(f"... {len(rows_all)} runs found under {RUNS_DIR}")

### 4.2 Seeds and the size of the noise

Every number in a results table is a draw from a distribution: the random initialization, the data
order, dropout masks and non-deterministic GPU kernels all change the outcome. Before you claim that
method A beats method B by one point you must know whether one point is inside the run-to-run noise.
The only way to find out is to repeat the run with different seeds and report **mean ± standard
deviation** (and the number of seeds). Three seeds are the minimum for a claim; five are standard;
for reinforcement learning ten are not excessive. The cell below trains the same CNN with several seeds
at a small budget and shows what the noise looks like; the models are kept so that Section 4.4 can
draw learning curves from them.

In [ ]:
SEEDS_V = [0, 1, 2] if FAST_RUN else [0, 1, 2, 3, 4]
tr_v, va_v = make_loaders(X_train, y_train, X_val, y_val, batch_size=32,
                          n_train=2000 if FAST_RUN else 10000, n_val=500 if FAST_RUN else 2000)
seed_trainers = []
for seed in SEEDS_V:
    seed_everything(seed)
    t = Trainer(SmallCNN(n_classes), TrainConfig(run_name=f"variance_seed{seed}", epochs=2 if FAST_RUN else 5,
                                                 lr=2e-3, patience=0, seed=seed), tr_v, va_v, verbose=False)
    t.fit()
    seed_trainers.append(t)
best_v = np.array([t.best_score for t in seed_trainers])
print(f"best val acc over {len(best_v)} seeds: mean {best_v.mean():.4f} ± {best_v.std(ddof=1):.4f} "
      f"(min {best_v.min():.4f}, max {best_v.max():.4f}, spread {best_v.max() - best_v.min():.4f})")
plt.bar([f"seed {s}" for s in SEEDS_V], best_v, color="C0", alpha=0.8)
plt.axhline(best_v.mean(), color="k", linestyle="--", label="mean")
plt.ylim(max(0, best_v.min() - 0.05), min(1, best_v.max() + 0.03)); plt.ylabel("best validation accuracy")
plt.title("The seed lottery: identical configuration, different seeds"); plt.legend(); plt.tight_layout(); plt.show()

The spread between the best and the worst seed is typically comparable to the improvements that
papers report for architectural tweaks. Whenever you compare two configurations, use the *same* set
of seeds for both and compare the pairs (Exercise 5 does this with a paired test), which removes the
shared variance and needs fewer runs to reach a conclusion.

### 4.3 Versions and the environment report

The third ingredient of reproducibility, after configurations and seeds, is the software environment.
Results can shift between PyTorch versions (default algorithms change), between CUDA and cuDNN
versions (different kernels), and between CPU and GPU. Record the versions with every run; the helper
below writes a small JSON file into the run directory and is cheap enough to call always. When the
project lives in a git repository, the commit hash of the code belongs in the same file.

In [ ]:
def environment_report(path=None):
    info = {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(),
            "platform": platform.platform(), "torch": torch.__version__, "numpy": np.__version__,
            "device": str(device), "cuda": torch.version.cuda if torch.cuda.is_available() else None,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "cudnn": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
            "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
            "seed": SEED, "fast_run": FAST_RUN}
    try:
        import torchvision
        info["torchvision"] = torchvision.__version__
    except ImportError:
        info["torchvision"] = None
    try:
        import subprocess
        out = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=5)
        info["git_commit"] = out.stdout.strip() or None
    except Exception:
        info["git_commit"] = None
    if path is not None:
        Path(path).write_text(json.dumps(info, indent=2), encoding="utf-8")
    return info


env_info = environment_report(t_cnn.run_dir / "environment.json")
print(json.dumps(env_info, indent=2))

### 4.4 Plots that communicate

A plot in a report has one job: to let the reader see the claim and its uncertainty in a few seconds.
Most training plots fail at this because they were made for the author during debugging. The pair
below shows the same data twice. On the left, the raw per-step loss of a single seed with no labels:
it is noisy, it hides the variance across seeds, and nobody can tell what the axes are. On the right,
the moving average of the same quantity, the mean over seeds with a shaded band for one standard
deviation, labelled axes and a title that states what is shown. The rules behind the second version:
average over seeds and show the spread; smooth per-step curves and say so; label axes with units;
use the same axis ranges when comparing panels; put the message in the title; and prefer one clear
plot over four cluttered ones.

In [ ]:
k = 10                                                                # moving-average window
L = min(len(t.step_losses) for t in seed_trainers)
curves = np.array([np.convolve(t.step_losses[:L], np.ones(k) / k, mode="valid") for t in seed_trainers])
mean_curve, std_curve = curves.mean(0), curves.std(0)
x = np.arange(len(mean_curve)) + k - 1

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(seed_trainers[0].step_losses[:L])
axes[0].set_title("before: one seed, raw, unlabelled")
axes[1].plot(x, mean_curve, label=f"mean of {len(curves)} seeds")
axes[1].fill_between(x, mean_curve - std_curve, mean_curve + std_curve, alpha=0.3, label="± 1 std")
axes[1].set_xlabel("optimizer step"); axes[1].set_ylabel(f"training loss (moving average, {k} steps)")
axes[1].set_title("after: SmallCNN training loss across seeds")
axes[1].legend()
plt.tight_layout(); plt.show()

### 4.5 Negative results and the report

Half of what you learn in a project is what did *not* work: the augmentation that hurt, the larger
model that overfit, the fancy optimizer that made no difference. Reports that omit this are less
useful and less credible. Put negative results in their own section with the same care as positive
ones (what was tried, at what budget, with how many seeds, and the numbers), and distinguish "did not
help" from "did not work because of a bug" from "was not tried".

Write the report in the order the template of Section 2.6 lists, but write the **abstract last** and
the **results table first**: the table forces you to decide what the claim is, and everything else
supports that claim. A reader should be able to reconstruct each table row from the run folders; the
"Reproducibility" section lists the run names and the command or cell that produced them. Keep the
main text short and move details to an appendix; a good project report is four to eight pages plus
figures.

## 5. What next

### 5.1 A reading list

**Books.** Goodfellow, Bengio and Courville, *Deep Learning* (2016) remains the reference for the
foundations of Parts I–II; Bishop and Bishop, *Deep Learning: Foundations and Concepts* (2024) is the
modern successor with a probabilistic viewpoint; Murphy, *Probabilistic Machine Learning: An
Introduction* (2022) and *Advanced Topics* (2023) are encyclopaedic; Prince, *Understanding Deep
Learning* (2023) is compact and up to date with free PDF and notebooks; Zhang, Lipton, Li and Smola,
*Dive into Deep Learning* pairs every topic with runnable code; Jurafsky and Martin, *Speech and
Language Processing* (3rd edition draft) for NLP; Sutton and Barto, *Reinforcement Learning: An
Introduction* (2nd edition, 2018) for Part VI.

**Courses.** Stanford CS231n (vision) and CS224n (NLP) have public lecture videos and assignments;
Karpathy's *Neural Networks: Zero to Hero* builds GPT from scratch in the spirit of Notebook 03 ·
Tensors and Autograd and Notebook 09 · Attention and Transformers; the fast.ai *Practical Deep
Learning for Coders* course is the best top-down complement to this bottom-up one; the Hugging Face
course covers the ecosystem of Notebook 10 · NLP with Pretrained Transformers (Hugging Face) in depth;
the DeepMind/UCL reinforcement-learning lectures extend Notebook 13 · Reinforcement Learning.

**Papers by area** (all real and worth reading in full). Vision: Krizhevsky et al. 2012 (AlexNet),
He et al. 2016 (ResNet), Dosovitskiy et al. 2021 (ViT), Liu et al. 2022 (ConvNeXt). Language:
Vaswani et al. 2017 (Transformer), Devlin et al. 2019 (BERT), Brown et al. 2020 (GPT-3), Hu et al.
2021 (LoRA), Ouyang et al. 2022 (InstructGPT), Rafailov et al. 2023 (DPO). Generative models: Kingma
and Welling 2014 (VAE), Goodfellow et al. 2014 (GAN), Ho et al. 2020 (DDPM), Rombach et al. 2022
(latent diffusion), Lipman et al. 2023 (flow matching). Reinforcement learning: Mnih et al. 2015
(DQN), Schulman et al. 2017 (PPO). Scaling and efficiency: Kaplan et al. 2020 and Hoffmann et al. 2022
(scaling laws, Chinchilla), Dao et al. 2022 (FlashAttention), Dettmers et al. 2023 (QLoRA). Trust:
Sundararajan et al. 2017 (Integrated Gradients), Selvaraju et al. 2017 (Grad-CAM), Madry et al. 2018
(adversarial training), Guo et al. 2017 (calibration), Lakshminarayanan et al. 2017 (deep
ensembles), Mitchell et al. 2019 (model cards), Gebru et al. 2021 (datasheets for datasets).

### 5.2 How to read a paper

Keshav's three-pass method (2007) is the practical recipe. **First pass** (ten minutes): title,
abstract, introduction, section headings, conclusions; decide whether the paper matters to you.
**Second pass** (an hour): read everything except proofs and appendices, look at every figure and
table carefully, note the references you do not know; you should be able to summarize the
contribution and its evidence. **Third pass** (several hours): re-implement the method in your head or
in code, challenge every assumption, and ask what you would have done differently. For machine
learning papers add a fourth check: find the baselines in the tables and ask whether they were tuned
as carefully as the proposed method, how many seeds were used, and whether the test set could have
been seen during development. The projects of Section 3 train exactly this habit.

### 5.3 Communities and staying current

The field moves quickly, but not as quickly as the announcements suggest. The arXiv listings for
cs.LG, cs.CV and cs.CL (and curated front ends such as Hugging Face Papers) are the primary sources;
the Hugging Face Hub and the
PyTorch forums are where practical questions get answered; Kaggle competitions are the fastest way to
learn what works on real data under time pressure; the main conferences (NeurIPS, ICML, ICLR, CVPR,
ACL) publish their proceedings openly, and their tutorials and workshops are often more useful than
the main track. Choose one or two sources and read them regularly rather than trying to follow
everything; re-implementing one paper a month teaches more than skimming a hundred abstracts.

### 5.4 Glossary

| Term | Meaning | Notebook |
|---|---|---|
| Activation function | Element-wise non-linearity (ReLU, GELU, sigmoid, tanh) between linear layers | 02 |
| Adam / AdamW | Adaptive optimizer with momentum and per-parameter scaling; AdamW decouples weight decay | 04 |
| Attention | Weighted average of values, with weights from query–key similarity | 09 |
| Autoencoder | Encoder–decoder trained to reconstruct its input through a bottleneck | 11 |
| Autograd | Automatic reverse-mode differentiation of a recorded computation graph | 03 |
| Backpropagation | The chain rule applied layer by layer to compute gradients of the loss | 02 |
| Baseline | The simplest model or heuristic a proposed model must beat | 17 |
| Batch normalization | Normalizes activations per channel with batch statistics; running averages at test time | 04 |
| Byte-Pair Encoding | Subword tokenizer learned by repeatedly merging the most frequent symbol pair | 10 |
| Calibration | Agreement between predicted probabilities and observed frequencies; measured by ECE | 16 |
| Checkpoint | Saved model/optimizer/scheduler state that allows resuming or restoring the best epoch | 05, 17 |
| Classifier-free guidance | Sampling with a mix of conditional and unconditional noise predictions to sharpen conditioning | 12 |
| Convolution | Local, weight-shared linear filter slid over an input; the core operation of CNNs | 06 |
| Cross-entropy | Negative log-likelihood of the correct class under the predicted softmax distribution | 01, 02 |
| Data augmentation | Label-preserving random transformations of inputs that act as regularization | 05, 06 |
| Data leakage | Information from the test set reaching training, e.g. duplicates or preprocessing fitted on all data | 17 |
| Diffusion model (DDPM) | Generative model that learns to reverse a gradual noising process | 12 |
| Decoder-only Transformer | Causally masked Transformer trained to predict the next token (GPT) | 09 |
| Distillation | Training a small student to match a large teacher's soft predictions | 15 |
| DPO | Direct Preference Optimization: preference learning as a logistic loss on policy/reference log-ratios | 10 |
| Dropout | Randomly zeroing activations during training to prevent co-adaptation | 04 |
| Early stopping | Halting training when the validation metric stops improving; restoring the best weights | 04, 17 |
| Embedding | Learned dense vector for a discrete symbol (token, class, position) | 08, 10 |
| ELBO | Evidence lower bound: the VAE objective (reconstruction minus KL to the prior) | 11 |
| Epoch / step | One pass over the training set / one optimizer update | 04 |
| Fine-tuning | Continuing training of a pretrained model on a downstream task | 07, 10 |
| FLOPs | Floating-point operations; the compute cost of a forward or training pass | 09, 14 |
| GAN | Generator and discriminator trained adversarially | 12 |
| Gradient clipping | Rescaling gradients whose norm exceeds a threshold to stabilize training | 04, 08 |
| Gradient descent (SGD) | Iteratively moving parameters against the (mini-batch) gradient of the loss | 01, 04 |
| Grad-CAM | Class-activation heatmap from gradients of a class score w.r.t. a convolutional feature map | 07, 16 |
| Hyperparameter | A setting chosen outside of training (learning rate, width, dropout rate) | 04, 14 |
| Inductive bias | Assumptions built into an architecture (locality for CNNs, sequence for RNNs) | 06, 07 |
| Integrated Gradients | Attribution by integrating input gradients along a path from a baseline | 16 |
| KL divergence | Asymmetric measure of the difference between two distributions | 01, 11 |
| KV cache | Storing the keys and values of previous tokens so that each generation step processes only the new token instead of recomputing the whole prefix | 09 |
| Label smoothing | Replacing one-hot targets by slightly soft targets to regularize | 04 |
| Layer normalization | Normalizing each token's activation vector; the default in Transformers | 04, 09 |
| Learning-rate schedule | Time-varying learning rate (warmup, cosine decay, one-cycle) | 04 |
| Logits | Unnormalized scores before the softmax | 02 |
| LoRA | Low-rank adaptation: trainable $BA$ update added to frozen weights | 10 |
| Loss function | Scalar objective minimized during training | 02 |
| LSTM / GRU | Gated recurrent cells that preserve gradients across long sequences | 08 |
| Mixed precision | Training with float16/bfloat16 activations and float32 master weights | 14 |
| Model card | Structured documentation of a model's intended use, data, metrics and limitations | 15, 17 |
| Multi-head attention | Several attention functions in parallel on projected subspaces | 09 |
| ONNX | Interchange format for exported models, run by ONNX Runtime | 15 |
| Overfitting / underfitting | Low training but high validation error / high error on both | 04 |
| Perplexity | Exponential of the average cross-entropy per token of a language model | 08, 09 |
| Policy gradient | Optimizing a stochastic policy by the gradient of expected return (REINFORCE, PPO) | 13 |
| Positional encoding | Injecting token positions into a permutation-equivariant Transformer | 09 |
| Pretraining | Large-scale self-supervised training that precedes task-specific fine-tuning | 07, 10 |
| Q-learning / DQN | Learning action values by bootstrapping from the Bellman optimality equation | 13 |
| Quantization | Representing weights/activations with low-bit integers for smaller, faster inference | 15 |
| Regularization / weight decay | Penalties or constraints that reduce overfitting | 04 |
| Reparameterization trick | Writing a sample as a deterministic function of noise so gradients flow | 11 |
| Residual connection | Adding a layer's input to its output; enables very deep networks | 06, 09 |
| RLHF | Reinforcement learning from human feedback: reward model plus KL-regularized policy optimization | 10, 13 |
| Scaling laws | Empirical power laws relating loss to parameters, data and compute | 09, 14 |
| Softmax | Maps logits to a probability distribution | 01 |
| Tokenization | Splitting text into vocabulary units (characters, words, subwords) | 10 |
| Transfer learning | Reusing a model trained on one task for another | 07 |
| Vanishing / exploding gradients | Gradients shrinking or growing exponentially with depth or time | 04, 08 |
| VAE | Variational autoencoder: latent-variable model trained with the ELBO | 11 |
| Weight initialization | Xavier/He schemes that keep activation and gradient variance stable across layers | 04 |

## Exercises

### Exercise 1 (warm-up): a one-page project plan

Pick one of the projects of Section 3 and write a one-page plan following the checklist of
Section 1.5: decision, primary metric with a target, secondary metrics, data (source, size, license,
split strategy), baselines, milestones with time estimates, the main risks, and the compute you have.
Represent the plan as a dataclass with a `to_markdown()` method, save it under `DATA_DIR/plans/`, and
print it.

**Solution to Exercise 1**

A dataclass forces every field to be filled in and makes the plan machine-readable, which is useful
when you keep several plans or want to diff them later. The example below is a plan for project (c);
replace the field values with your own.

In [ ]:
@dataclass
class ProjectPlan:
    title: str
    decision: str
    primary_metric: str
    target: str
    data: str
    split: str
    compute: str
    secondary_metrics: list = field(default_factory=list)
    baselines: list = field(default_factory=list)
    milestones: list = field(default_factory=list)
    risks: list = field(default_factory=list)

    def to_markdown(self):
        bullets = lambda items: "\n".join(f"- {x}" for x in items)
        return (f"# Project plan: {self.title}\n\n"
                f"**Decision supported.** {self.decision}\n\n"
                f"**Primary metric and target.** {self.primary_metric}; target: {self.target}\n\n"
                f"**Secondary metrics.**\n{bullets(self.secondary_metrics)}\n\n"
                f"**Data.** {self.data}\n\n**Split.** {self.split}\n\n"
                f"**Baselines.**\n{bullets(self.baselines)}\n\n"
                f"**Milestones.**\n{bullets(self.milestones)}\n\n"
                f"**Risks.**\n{bullets(self.risks)}\n\n**Compute.** {self.compute}\n")


plan = ProjectPlan(
    title="Hourly electricity-load forecasting (project c)",
    decision="Schedule next-day generation: forecasts 1-24 h ahead drive the unit-commitment plan.",
    primary_metric="MAE at horizon 24 h in MW on the held-out last 3 months",
    target="MASE < 0.8 relative to the seasonal-naive forecast (24 h)",
    data="UCI Electricity Load Diagrams, 370 clients, 15-min resolution aggregated to hourly (~250 MB, CC BY 4.0)",
    split="chronological: train 2012-2013, validation Jan-Sep 2014, test Oct-Dec 2014; scaler fitted on train only",
    compute="one laptop GPU (6 GB); each run < 10 min",
    secondary_metrics=["MAE at 1 h and 12 h", "coverage of 90% prediction intervals (stretch)", "training time"],
    baselines=["naive (last value)", "seasonal naive (24 h and 168 h)", "linear model on the 168-h window"],
    milestones=["week 1: audit, split, baselines, results table skeleton",
                "week 2: GRU and Transformer forecasters, 3 seeds, error-vs-horizon plot",
                "week 3: ablations (context length, direct vs autoregressive, per-window normalization)",
                "week 4: error analysis by hour-of-day and weekday, report and model card"],
    risks=["holidays and missing days break the seasonal pattern (need calendar features)",
           "the Transformer may not beat the linear baseline at this scale (report it either way)"])
plans_dir = DATA_DIR / "plans"
plans_dir.mkdir(parents=True, exist_ok=True)
plan_md = plan.to_markdown()
(plans_dir / "forecasting_plan.md").write_text(plan_md, encoding="utf-8")
print(plan_md)

### Exercise 2: run a baseline properly

Using the `Trainer`, add a one-hidden-layer MLP between the linear model and the CNN of Section 2.3,
train it with the same data and budget, and produce a results table in which every row also carries a
95% bootstrap confidence interval of the validation accuracy (use `bootstrap_ci` from the project (e)
starter on the per-item correctness from `predict`). Which differences are outside the intervals?

**Solution to Exercise 2**

The interval width depends only on the number of validation items and the accuracy, so with the
1,000 validation items of the `FAST_RUN` budget every interval is roughly ±2–3 points wide (about
±1 point with the 5,000 items of the full run); differences smaller than that between two rows are
not established by a single run.

In [ ]:
seed_everything(SEED)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, n_classes))
cfg_mlp = TrainConfig(run_name="baseline_mlp", epochs=min(EPOCHS_DEMO, 5), lr=1e-3, patience=0, notes="1 hidden layer, 256 units")
t_mlp = Trainer(mlp, cfg_mlp, train_loader, val_loader, verbose=False)
t_mlp.fit()

baseline_trainers = {"linear": t_linear, "MLP": t_mlp, "small CNN": t_cnn}
print(f"{'model':12s} {'params':>9s} {'val acc':>8s} {'95% CI':>18s}")
for name, tr in baseline_trainers.items():
    tr.load_checkpoint(tr.run_dir / "best.pt", model_only=True)        # evaluate the best epoch
    outs, targets = tr.predict(val_loader)
    correct = (outs.argmax(1) == targets).cpu().numpy()
    m, lo, hi = bootstrap_ci(correct)
    print(f"{name:12s} {count_params(tr.model):9,d} {m:8.3f}   [{lo:.3f}, {hi:.3f}]")

### Exercise 3: write a model card

Write a function `model_card(trainer, data_source, class_names)` that fills the model-card
template (Mitchell et al., 2019; also used in Notebook 15 · Deployment and Model Optimization) from
the trainer's configuration, history and a fresh evaluation, including per-class accuracy, and saves
it as `MODEL_CARD.md` in the run directory. Apply it to the CNN.

**Solution to Exercise 3**

Everything factual in the card is generated from the run artefacts, so the card cannot drift away
from the model it describes; the sections that need human judgement (intended use, ethical
considerations) are filled with explicit placeholders that the author must edit, and the function
says so.

In [ ]:
def model_card(trainer, data_source, class_names, author="course participant"):
    cfg = trainer.cfg
    best = (max if cfg.mode == "max" else min)(trainer.history, key=lambda r: r[cfg.monitor])
    outs, targets = trainer.predict(trainer.val_loader)
    preds = outs.argmax(1)
    per_class = [(c, ((preds == i) & (targets == i)).sum().item() / max(1, (targets == i).sum().item()))
                 for i, c in enumerate(class_names)]
    per_class_md = "\n".join(f"| {c} | {a:.3f} |" for c, a in per_class)
    md = f"""# Model card: {cfg.run_name}

## Model details
- Developed by: {author}, {time.strftime('%Y-%m-%d')}
- Architecture: {type(trainer.model).__name__} with {count_params(trainer.model):,} parameters
- Training: {cfg.optimizer}, lr {cfg.lr}, weight decay {cfg.weight_decay}, {cfg.scheduler} schedule, batch {cfg.batch_size}, {len(trainer.history)} epochs (budget {cfg.epochs}), seed {cfg.seed}
- Notes: {cfg.notes or '-'}
- Artefacts: {trainer.run_dir} (config.json, log.jsonl, best.pt, environment.json)

## Intended use
- Primary use: classification of {len(class_names)}-class {data_source} images (EDIT: describe the deployment context)
- Out of scope: other image domains, resolutions or class sets; safety-critical decisions

## Training and evaluation data
- Source: {data_source}; training examples: {len(trainer.train_loader.dataset):,}; validation examples: {len(trainer.val_loader.dataset):,}
- Preprocessing: pixel values scaled to [0, 1]; no augmentation

## Metrics
- Monitored metric: {cfg.monitor} ({cfg.mode}); best value {best[cfg.monitor]:.4f} at epoch {best['epoch']}
- Validation loss at best epoch: {best['val_loss']:.4f}

## Quantitative analysis (per-class accuracy on validation)
| Class | Accuracy |
|---|---|
{per_class_md}

## Ethical considerations and limitations
- EDIT: known biases of the data, failure modes seen in error analysis, calibration status
- The reported numbers come from a single seed; see the variance study before drawing conclusions
"""
    (trainer.run_dir / "MODEL_CARD.md").write_text(md, encoding="utf-8")
    return md


print(model_card(t_cnn, data_source, class_names))

### Exercise 4: extend the `Trainer` with a hook

Subclass `Trainer` to record the gradient norm at every optimizer step (the value returned by
`clip_grad_norm_`) and to print its mean and maximum at the end of each epoch through the
`on_epoch_end` hook. Train the CNN for a short budget and plot the gradient norm over steps. What
would a sudden spike, or a norm that stays pinned at the clipping threshold, tell you?

**Solution to Exercise 4**

Only `train_step` and `on_epoch_end` are overridden; everything else (logging, checkpoints, early
stopping) is inherited. A norm that sits at the clipping threshold for most steps means the threshold
is too low (clipping has become the effective learning-rate schedule); isolated spikes point to
outlier batches or a learning rate near the edge of stability; a norm that decays to near zero while
the loss is still high suggests saturation or dead units.

In [ ]:
class GradNormTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.grad_norms = []

    def train_step(self, xb, yb):
        logits = self.model(xb)
        loss = self.loss_fn(logits, yb)
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        total_norm = nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip or float("inf"))
        self.grad_norms.append(total_norm.item())
        self.optimizer.step()
        return loss.item()

    def on_epoch_end(self, row):
        recent = self.grad_norms[-len(self.train_loader):]
        self._say(f"   gradient norm: mean {np.mean(recent):.3f} | max {np.max(recent):.3f} | "
                  f"clipped on {np.mean(np.array(recent) > self.cfg.grad_clip):.0%} of steps")


seed_everything(SEED)
t_grad = GradNormTrainer(SmallCNN(n_classes), TrainConfig(run_name="gradnorm_demo", epochs=2 if FAST_RUN else 4,
                                                          lr=2e-3, grad_clip=1.0, patience=0), tr_v, va_v)
t_grad.fit()
plt.plot(t_grad.grad_norms, linewidth=0.8)
plt.axhline(t_grad.cfg.grad_clip, color="r", linestyle="--", label="clip threshold")
plt.xlabel("optimizer step"); plt.ylabel("gradient norm (before clipping)"); plt.title("Gradient norm during training")
plt.legend(); plt.tight_layout(); plt.show()

### Exercise 5 (challenge): a paired comparison over seeds

Compare two configurations of the CNN (AdamW versus SGD with momentum, at learning rates that are
reasonable for each) using the *same* seeds for both, with `run_grid`. Report mean ± std for each,
the mean paired difference with its standard error, a paired t-test (use `scipy.stats.ttest_rel` if
available, otherwise compute the t statistic by hand) and Cohen's $d$ of the paired differences. Is
the difference significant at this budget? What would you need to change to make the test more
powerful?

**Solution to Exercise 5**

Pairing by seed removes the variance that both configurations share (the data subset, the
initialization pattern), so the paired test is more powerful than comparing two independent groups.
With three seeds and one epoch the test rarely reaches significance, which is the correct outcome: the
power of the test grows with the number of seeds and with the size of the effect relative to the
noise, so the two levers are more seeds and a budget at which the methods actually differ. Two
caveats belong in any report that uses this test. The paired t-test assumes that the per-seed
differences are independent draws from a roughly normal distribution, which cannot be checked with
three to five seeds, so quote the effect size (Cohen's $d$ of the differences) and the raw differences
next to the p-value and treat p-values from such tiny samples as indicative rather than decisive.
And the seeds share the initialization and the data subset but not the mini-batch order (the
DataLoader's generator advances across runs), so pairing removes some, not all, of the shared variance.

In [ ]:
SEEDS_E5 = [0, 1, 2] if FAST_RUN else [0, 1, 2, 3, 4]
cfg_adamw = TrainConfig(run_name="ex5_adamw", epochs=1 if FAST_RUN else 5, lr=2e-3, optimizer="adamw", patience=0)
cfg_sgd = TrainConfig(run_name="ex5_sgd", epochs=1 if FAST_RUN else 5, lr=5e-2, optimizer="sgd", patience=0)
rows_e5 = run_grid({"AdamW": (lambda: SmallCNN(n_classes), cfg_adamw), "SGD+momentum": (lambda: SmallCNN(n_classes), cfg_sgd)},
                   seeds=SEEDS_E5, n_train=1000 if FAST_RUN else 10000, n_val=500 if FAST_RUN else 2000)
summary_e5 = summarize_grid(rows_e5)

a = np.array([r["best"] for r in rows_e5 if r["config"] == "AdamW"])
b = np.array([r["best"] for r in rows_e5 if r["config"] == "SGD+momentum"])
diff = a - b
n = len(diff)
se = diff.std(ddof=1) / math.sqrt(n)
t_stat = diff.mean() / se if se > 0 else float("inf")
cohen_d = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else float("inf")
try:
    from scipy import stats
    p_value = stats.ttest_rel(a, b).pvalue
except ImportError:
    p_value = None
print(f"\npaired difference AdamW - SGD: {diff.mean():+.4f} ± {se:.4f} (s.e.), n = {n} seeds")
print(f"t = {t_stat:.2f}, Cohen's d = {cohen_d:.2f}" + (f", p = {p_value:.3f}" if p_value is not None else " (scipy unavailable: no p-value)"))
verdict = "significant at the 5% level" if (p_value is not None and p_value < 0.05) else "not significant at this budget"
print("conclusion:", verdict)

## Summary

- A deep learning project starts with a hypothesis: the decision to support, one primary metric with a
  target, baselines to beat and the constraints; write it down before writing model code.
- Audit the data (balance, duplicates, leakage, label quality) and compute trivial baselines first;
  every later result is judged against them.
- Reuse one tested training harness. The `Trainer` of this notebook covers configuration, history,
  JSONL logging, checkpointing with resume, pluggable loss and metrics, early stopping and hooks in
  about a hundred lines; adapt it rather than rewriting loops per experiment.
- Iterate one change at a time, keep an experiment log, and let error analysis (confusion matrix,
  most confident mistakes, per-group metrics) decide the next step.
- Each of the six project briefs (transfer learning, word-level Transformer LM, forecasting,
  conditional diffusion, LoRA fine-tuning, reinforcement learning) plus the optional reproduction has
  milestones ordered so that stopping after any of them yields a reportable result.
- Results are draws from a distribution: report mean ± std over seeds, compare configurations on the
  same seeds with paired statistics, and quote confidence intervals for test-set metrics.
- Reproducibility means run folders with configs, logs and checkpoints, an environment report with
  versions, seeds, and commands that regenerate every table row.
- Reports follow a fixed structure (abstract, data, method, experiments, analysis, limitations,
  reproducibility), include negative results, use plots that show the claim and its uncertainty, and
  ship with a model card.

## Further reading

- Keshav, 2007, *How to Read a Paper* (the three-pass method).
- Pineau et al., 2021, *Improving Reproducibility in Machine Learning Research: A Report from the NeurIPS 2019 Reproducibility Program* (and the ML Code Completeness Checklist).
- Bouthillier et al., 2021, *Accounting for Variance in Machine Learning Benchmarks*.
- Henderson et al., 2018, *Deep Reinforcement Learning that Matters* (seed variance in RL).
- Mitchell et al., 2019, *Model Cards for Model Reporting*; Gebru et al., 2021, *Datasheets for Datasets*.
- Sculley et al., 2015, *Hidden Technical Debt in Machine Learning Systems*.
- Karpathy, 2019, *A Recipe for Training Neural Networks* (blog post; the "overfit one batch first" discipline).
- Tufte, 2001, *The Visual Display of Quantitative Information* (2nd edition), for plots that communicate.
- Goodfellow, Bengio and Courville, 2016, *Deep Learning*, Chapter 11, *Practical Methodology*.
- PyTorch documentation, *Reproducibility* and *Saving and Loading Models*: https://docs.pytorch.org/docs/stable/